# VerifiQA — paper results (single source of truth)

Every figure and table below is computed directly from the run files in `results/` on the deterministic-SMT runs. No numbers are hardcoded; re-running the notebook regenerates all artifacts. Mapping to the paper:

| Notebook section | Paper artifact |
|---|---|
| Main results (both datasets) | Table `tab:main` |
| Acceptance outcomes figure | Figure `fig:acceptance-outcomes` |
| Precision & abstention figure | Supporting/appendix figure |
| VeriFin across answer models | Figure `fig:model-robustness` |
| Deterministic vs. autoformalized SMT | Ablation table |
| Coverage & solver diagnostics | Table `tab:abstention` + Results prose |
| Dataset composition | Table `tab:topics` |
| Per-metric coverage | Appendix |
| Metric registry coverage | Appendix |
| Repair ablation | Appendix (optional) |


## Table 1 — Main results (both datasets)

Precision = share of accepted answers correct (soundness); FA = false accepts; Accuracy = share of decided questions decided correctly; Abstention = share declined. VerifiQA is the only method with FA=0 on either dataset.

## Across answer models (both datasets) — `tab:models`

All FinanceBench metrics use the complete 67-question population. Counts are
recomputed from per-question artifacts with candidate-specific adjudications.
GPT-5.5 has an auditable XBRLBench run; no FinanceBench GPT-5.5 artifact is
available and that entry must remain missing.

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
nan=float("nan")

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])

def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out

def _rid(r): return r.get("id") or r.get("financebench_id")

def _ok(rid, cv, g, kc):
    adjudicated = _adjudicated_ok(rid, cv) if kc else _NO_ADJUDICATION
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    if cv is None: return None
    t=_nums(g)
    return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None

def _parseable_ids(path):
    """Answerable questions: those with a parseable numeric candidate, measured from the
    raw-LLM baseline (which parses every candidate). Questions with no numeric candidate
    are candidate-generation failures, shared by all methods and excluded from abstention."""
    ids=set()
    for line in Path(path).read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line)
        if r.get("claimed_value") is not None: ids.add(_rid(r))
    return ids

def _score(path, kc, parseable):
    TA=TC=FR=FA=0; status={}
    for line in Path(path).read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line); rid=_rid(r); st=r.get("status"); cv=r.get("claimed_value")
        status[rid]=st
        if cv is None: continue                      # no candidate: not a decision
        o=_ok(rid, cv, r.get("gold_answer") or r.get("answer"), kc)
        if st=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif st=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)
    ver=TA+FA; dec=ver+TC+FR
    # Abstention = AB/n over all benchmark questions, matching the paper's metric
    # definition. Questions with no parseable candidate count as declined.
    nd_all=sum(1 for st in status.values() if st not in ("VERIFIED","VIOLATED"))
    return {"Precision": 100*TA/ver if ver else nan,
            "Accuracy": 100*(TA+TC)/dec if dec else nan,
            "Abstention": 100*nd_all/len(status) if status else nan,
            "FA": FA}

DATASETS = [
    ("XBRLBench", False, {
        "raw-LLM":"onepool/xbrl_none", "LLM-judge":"onepool/xbrl_judge",
        "judge+formula":"onepool/xbrl_judge_formula", "PoT":"onepool/xbrl_pot",
        "VeriFin (ours)":"mc_calc_detsmt_v2_haiku"}),
    ("FinanceBench", True, {
        "raw-LLM":"onepool/fb_none", "LLM-judge":"onepool/fb_judge",
        "judge+formula":"onepool/fb_judge_formula", "PoT":"onepool/fb_pot",
        "VeriFin (ours)":"fb_detsmt_v2_haiku"}),
]
METHODS = ["raw-LLM","LLM-judge","judge+formula","PoT","VeriFin (ours)"]

cols={}
answerable={}
for dset, kc, runs in DATASETS:
    parseable=_parseable_ids(ROOT/"results"/runs["raw-LLM"]/"results.jsonl")
    answerable[dset]=len(parseable)
    for m in METHODS:
        s=_score(ROOT/"results"/runs[m]/"results.jsonl", kc, parseable)
        for metric,val in s.items():
            cols.setdefault((dset,metric),{})[m]=val

main_results = pd.DataFrame(cols).reindex(METHODS)
main_results.columns = pd.MultiIndex.from_tuples(main_results.columns)
print("answerable (parseable-candidate) questions:", answerable)
def _fmt(v,c): return f"{int(v)}" if c[1]=="FA" else f"{v:.1f}"
print(pd.DataFrame({c:[_fmt(v,c) for v in main_results[c]] for c in main_results.columns},
                   index=METHODS).to_string())

# ---- paper LaTeX (tab:main), emitted from the same numbers ----
def _cell(m, dset, metric, bold_if_best=False):
    v=main_results.loc[m,(dset,metric)]
    s=f"{int(v)}" if metric=="FA" else f"{v:.1f}"
    return s
best={("XBRLBench","Precision"),("XBRLBench","Accuracy"),("XBRLBench","FA"),
      ("FinanceBench","Precision"),("FinanceBench","FA")}   # cells VeriFin wins
lines=[]
for m in METHODS:
    cells_=[]
    for dset in ["XBRLBench","FinanceBench"]:
        for metric in ["Precision","Accuracy","Abstention","FA"]:
            s=_cell(m,dset,metric)
            if m=="VeriFin (ours)" and (dset,metric) in best: s=f"\\bfseries {s}"
            cells_.append(s)
    row="  & "+" & ".join(cells_)+" \\\\"
    if m=="VeriFin (ours)": row="\\rowcolor{ourshl}\n"+m+row
    else: row=m+row
    lines.append(row)
print("\n% ---- tab:main LaTeX (auditable, from result files) ----")
print("\n".join(lines))
main_results


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.paper_scoring import load_manual_adjudications, load_policy, score_all

policy_path = ROOT / "results/scoring_policy.json"
if not policy_path.is_file():
    policy_path = ROOT / "publication_assets_source/results/scoring_policy.json"
policy = load_policy(policy_path)
adjudications = load_manual_adjudications(
    policy_path.parent / policy["candidate_correctness"]["manual_adjudications"]
)
scored = {
    (row["dataset"], row["run"]): row
    for row in score_all(ROOT, policy, adjudications)
    if row["group"] == "answer_models"
}

MODEL_ORDER = [
    ("GPT-5.5", "gpt-5.5"),
    ("Claude Haiku 4.5", "claude-haiku-4.5"),
    ("Fin-o1", "fino1-identity-pending"),
    ("Qwen3-30B", "qwen3-30b"),
    ("Llama-3.1-8B", "llama-3.1-8b"),
    ("Qwen2.5-7B", "qwen2.5-7b"),
]

def _model_metrics(dataset, run):
    row = scored[(dataset, run)]
    if row["status"] == "missing":
        return {"Prec": None, "FA": None, "FR": None, "Abst": None}
    accepted = row["TA"] + row["FA"]
    n = policy["datasets"][dataset]["denominator"]
    return {
        "Prec": round(100 * row["TA"] / accepted, 1) if accepted else float("nan"),
        "FA": row["FA"],
        "FR": row["FR"],
        "Abst": round(100 * row["AB"] / n, 1),
    }

rows = []
for label, run in MODEL_ORDER:
    xbrl = _model_metrics("xbrlfiling", run)
    finance = _model_metrics("financebench", run)
    rows.append({
        "Answer model": label,
        ("XBRLFiling", "Prec"): xbrl["Prec"],
        ("XBRLFiling", "FA"): xbrl["FA"],
        ("XBRLFiling", "FR"): xbrl["FR"],
        ("XBRLFiling", "Abst"): xbrl["Abst"],
        ("FinanceBench", "Prec"): finance["Prec"],
        ("FinanceBench", "FA"): finance["FA"],
        ("FinanceBench", "FR"): finance["FR"],
        ("FinanceBench", "Abst"): finance["Abst"],
    })

models_both = pd.DataFrame(rows).set_index("Answer model")
models_both.columns = pd.MultiIndex.from_tuples(models_both.columns)
print(models_both.to_string())

def _cell(value, count=False):
    if value is None or pd.isna(value):
        return "{--}"
    return f"{int(value)}" if count else f"{float(value):.1f}"

print("\n% ---- tab:models LaTeX (recomputed from canonical result files) ----")
for label, _ in MODEL_ORDER:
    row = models_both.loc[label]
    cells = []
    for dataset in ("XBRLFiling", "FinanceBench"):
        cells.extend([
            _cell(row[(dataset, "Prec")]),
            _cell(row[(dataset, "FA")], count=True),
            _cell(row[(dataset, "FR")], count=True),
            _cell(row[(dataset, "Abst")]),
        ])
    print(label.replace(" ", "~") + " & " + " & ".join(cells) + r" \\")

models_both


## Across all answer models — `tab:models-full`

Full outcome breakdown (TA/FA/TR/FR/AB) with precision, decision accuracy, and
coverage over all 600 XBRLBench and all 67 FinanceBench questions. The canonical
scorer is `scripts/paper_scoring.py`; FinanceBench GPT-5.5 remains unavailable.

In [ ]:
# tab:models-full — VeriFin outcome breakdown across all answer models.
# TA/FA/TR/FR/AB counts + precision, accuracy, coverage, computed from results/*.
import json, re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])
# FinanceBench is scored over all 67 questions, matching the GPT-5.5 run.
# Caveat: 5 questions ask for a description/entity and their gold answers are
# prose. String-extracted numbers from those golds are not the quantity a numeric
# verifier computes -- e.g. 01936's gold "87% of the total restructuring
# liability..." is a percentage, while the verifier correctly grounds the total
# liability as $93M. Llama's single FA is this artifact, not a wrong value.
EXCLUDE_FB = set()

def _rid(r): return r.get("id") or r.get("financebench_id")

def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out

def _ok(r):
    adjudicated = _adjudicated_ok(_rid(r), r.get("claimed_value"))
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    cv = r.get("claimed_value")
    if cv is None: return None
    t = _nums(r.get("gold_answer") or r.get("answer"))
    return any(abs(abs(cv)-x) <= max(1.0, x*0.01) for x in t) if t else None

def _score(run, exclude):
    rows = [json.loads(l) for l in (ROOT/"results"/run/"results.jsonl").read_text().splitlines() if l.strip()]
    rows = [r for r in rows if _rid(r) not in exclude]
    TA=FA=TR=FR=AB=0
    for r in rows:
        s, o = r.get("status"), _ok(r)
        if   s == "VERIFIED": TA,FA = (TA+1,FA) if o else (TA,FA+1)
        elif s == "VIOLATED": TR,FR = (TR+1,FR) if (o is False or o is None) else (TR,FR+1)
        else:                 AB += 1
    n, dec, ver = len(rows), TA+FA+TR+FR, TA+FA
    return {"n":n,"TA":TA,"FA":FA,"TR":TR,"FR":FR,"AB":AB,
            "Precision": round(100*TA/ver,1) if ver else float("nan"),
            "Accuracy":  round(100*(TA+TR)/dec,1) if dec else float("nan"),
            "Coverage":  round(100*dec/n,1) if n else float("nan")}

# (model, XBRLFiling run, FinanceBench run or None)
MODELS = [
    ("Llama-3.1-8B",     "mc_calc_detsmt_v2_llama3.1-8b", "fb_detsmt_v2_llama3.1-8b"),
    ("Qwen2.5-7B",       "mc_calc_detsmt_v2_qwen2.5-7b",  "fb_detsmt_qwen2.5-7b"),
    ("Qwen3-30B",        "latest/mc-qwen-30b-latest",
                         "latest/fb_qwen3-30b_updated2_det_smt_run3_new"),
    ("Fin-o1",       "latest/mc-fino-30b-latest",     "latest/finbench-fino-30b-latest"),
    ("Claude Haiku 4.5", "mc_calc_detsmt_v2_haiku",       "fb_detsmt_v2_haiku"),
    ("GPT-5.5",          "latest/mc_calc_gpt5.5_codex_det_smt_Nrepair_run1", None),  # no FinanceBench run
]

rows=[]
for name, xrun, frun in MODELS:
    for ds, run, exc in [("XBRLFiling", xrun, set()), ("FinanceBench", frun, EXCLUDE_FB)]:
        rows.append({"Answer model":name, "Dataset":ds,
                     **(_score(run, exc) if run else
                        {k: None for k in ("n","TA","FA","TR","FR","AB","Precision","Accuracy","Coverage")})})
models_full = pd.DataFrame(rows).sort_values(["Dataset","TA"], ascending=[False,True])
print(models_full.to_string(index=False))

pool = models_full.dropna(subset=["TA"])
acc  = int((pool.TA + pool.FA).sum()); fa = int(pool.FA.sum())
x    = pool[pool.Dataset=="XBRLFiling"]
print(f"\nPooled: {acc} accepted, {fa} false accept(s). "
      f"XBRLFiling: {int((x.TA+x.FA).sum())} accepted, {int(x.FA.sum())} false accepts.")

print("\n% ---- tab:models-full LaTeX body ----")
def _c(v, dec=False):
    if v is None or (isinstance(v,float) and pd.isna(v)): return "{--}"
    return f"{v:.1f}" if dec else f"{int(v)}"
for ds in ["XBRLFiling","FinanceBench"]:
    print(f"% {ds}")
    for _, r in models_full[models_full.Dataset==ds].iterrows():
        cells = [_c(r.TA), _c(r.FA), _c(r.TR), _c(r.FR), _c(r.AB),
                 _c(r.Precision,1), _c(r.Accuracy,1), _c(r.Coverage,1)]
        print(f"{r['Answer model']:17} & " + " & ".join(cells) + r" \\")

models_full


## Main-paper figure — Risk–coverage operating points

Coverage is end-to-end: all 600 XBRLBench questions and all 67 FinanceBench
questions are retained. Each method contributes one measured operating point;
the markers are intentionally unconnected because the runs contain no
confidence score or threshold sweep from which to estimate a genuine curve.


In [ ]:
# ICAIF main-paper figure: observed risk--coverage operating points.
#
# x = decision coverage = (VERIFIED + VIOLATED) / all benchmark questions
# y = accepted-answer false-accept risk = FA / (TA + FA) = 1 - precision
#
# FinanceBench uses all 67 questions. A row without a parseable numerical
# candidate is uncovered and therefore lowers end-to-end decision coverage.
# Baselines use the canonical one-pool shared-answer runs.
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D


def _risk_policy_path(root):
    for relative_path in (
        "publication_assets_source/results/scoring_policy.json",
        "results/scoring_policy.json",
    ):
        candidate = root / relative_path
        if candidate.is_file():
            return candidate
    return None


def _risk_coverage_root():
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents, cwd / "verifiqa", Path("/content/verifiqa")]
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / "scripts/paper_scoring.py").is_file()
            and _risk_policy_path(candidate) is not None
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate scripts/paper_scoring.py and the canonical scoring "
        "policy. Set the notebook working directory to a private or public "
        "VeriFin repository root."
    )


RISK_COVERAGE_ROOT = _risk_coverage_root()


def _load_risk_scoring(root):
    module_path = root / "scripts/paper_scoring.py"
    spec = importlib.util.spec_from_file_location(
        "verifin_paper_scoring_for_risk_coverage", module_path
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load the canonical scorer from {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    policy_path = _risk_policy_path(root)
    if policy_path is None:
        raise FileNotFoundError(f"No canonical scoring policy found under {root}")
    policy = module.load_policy(policy_path)
    manual_name = policy["candidate_correctness"].get(
        "manual_adjudications", "manual_adjudications.json"
    )
    adjudications = module.load_manual_adjudications(
        policy_path.parent / manual_name
    )
    return module, policy, adjudications


RISK_SCORING, RISK_POLICY, RISK_ADJUDICATIONS = _load_risk_scoring(
    RISK_COVERAGE_ROOT
)
RISK_COVERAGE_RUNS = {
    "XBRLBench": {
        "raw-LLM": (
            "baselines", "direct-llm", "results/onepool/xbrl_none"
        ),
        "LLM-judge": (
            "baselines", "llm-judge", "results/onepool/xbrl_judge"
        ),
        "judge+formula": (
            "baselines",
            "judge-plus-formula",
            "results/onepool/xbrl_judge_formula",
        ),
        "PoT": (
            "baselines", "program-of-thought", "results/onepool/xbrl_pot"
        ),
        "VeriFin": (
            "answer_models",
            "claude-haiku-4.5",
            "results/mc_calc_detsmt_v2_haiku",
        ),
    },
    "FinanceBench": {
        "raw-LLM": (
            "baselines", "direct-llm", "results/onepool/fb_none"
        ),
        "LLM-judge": (
            "baselines", "llm-judge", "results/onepool/fb_judge"
        ),
        "judge+formula": (
            "baselines",
            "judge-plus-formula",
            "results/onepool/fb_judge_formula",
        ),
        "PoT": (
            "baselines", "program-of-thought", "results/onepool/fb_pot"
        ),
        "VeriFin": (
            "answer_models",
            "claude-haiku-4.5",
            "results/fb_detsmt_v2_haiku",
        ),
    },
}
DATASET_KEYS = {"XBRLBench": "xbrlfiling", "FinanceBench": "financebench"}
DATASET_TOTALS = {"XBRLBench": 600, "FinanceBench": 67}
METHOD_ORDER = ["raw-LLM", "LLM-judge", "judge+formula", "PoT", "VeriFin"]

RISK_DATASET_IDS = {}
for _dataset_label, _dataset_key in DATASET_KEYS.items():
    _dataset_spec = RISK_POLICY["datasets"][_dataset_key]
    _dataset_rows = RISK_SCORING.load_jsonl(
        RISK_COVERAGE_ROOT / _dataset_spec["source_path"]
    )
    RISK_DATASET_IDS[_dataset_key] = RISK_SCORING.validate_dataset_ids(
        _dataset_rows, _dataset_key, _dataset_spec
    )


def _risk_coverage_point(dataset, method, run):
    dataset_key = DATASET_KEYS[dataset]
    group, run_name, expected_source = run
    run_spec = RISK_POLICY["canonical_runs"][dataset_key][group][run_name]
    policy_source = run_spec.get("source_path")
    if policy_source != expected_source:
        raise AssertionError(
            f"{dataset}/{run_name}: policy source {policy_source!r} does not "
            f"match the figure source {expected_source!r}."
        )

    rows = RISK_SCORING.load_jsonl(
        RISK_COVERAGE_ROOT / policy_source / "results.jsonl"
    )
    RISK_SCORING.validate_run_ids(
        rows, RISK_DATASET_IDS[dataset_key], dataset_key, run_name
    )
    counts = RISK_SCORING.score_rows(
        rows, dataset_key, RISK_POLICY, RISK_ADJUDICATIONS
    )
    declared = run_spec.get("counts")
    if counts != declared:
        raise AssertionError(
            f"{dataset}/{run_name}: canonical scorer returned {counts}, but "
            f"the policy declares {declared}."
        )

    expected_total = RISK_POLICY["datasets"][dataset_key]["denominator"]
    if expected_total != DATASET_TOTALS[dataset]:
        raise AssertionError(
            f"{dataset}: policy denominator {expected_total} does not match "
            f"the figure denominator {DATASET_TOTALS[dataset]}."
        )
    if sum(counts.values()) != expected_total:
        raise AssertionError(f"{dataset}/{run_name}: outcomes do not sum to N.")

    decided = counts["TA"] + counts["FA"] + counts["TR"] + counts["FR"]
    accepted = counts["TA"] + counts["FA"]
    return {
        "Dataset": dataset,
        "Method": method,
        "Questions": expected_total,
        "Decided": decided,
        "Accepted": accepted,
        "TA": counts["TA"],
        "FA": counts["FA"],
        "Coverage": 100.0 * decided / expected_total,
        "False-accept risk": (
            100.0 * counts["FA"] / accepted if accepted else float("nan")
        ),
    }


risk_coverage_points = pd.DataFrame(
    [
        _risk_coverage_point(dataset, method, run)
        for dataset, runs in RISK_COVERAGE_RUNS.items()
        for method, run in runs.items()
    ]
)

# Lock the plotted points to the auditable main-table results.
EXPECTED_POINTS = {
    ("XBRLBench", "raw-LLM"): (100.0, 15.3, 92),
    ("XBRLBench", "LLM-judge"): (100.0, 11.8, 26),
    ("XBRLBench", "judge+formula"): (100.0, 13.5, 75),
    ("XBRLBench", "PoT"): (96.3, 1.3, 6),
    ("XBRLBench", "VeriFin"): (98.8, 0.0, 0),
    ("FinanceBench", "raw-LLM"): (95.5, 28.1, 18),
    ("FinanceBench", "LLM-judge"): (95.5, 24.1, 7),
    ("FinanceBench", "judge+formula"): (95.5, 27.7, 13),
    ("FinanceBench", "PoT"): (95.5, 9.1, 4),
    ("FinanceBench", "VeriFin"): (80.6, 0.0, 0),
}
for row in risk_coverage_points.to_dict("records"):
    expected = EXPECTED_POINTS[(row["Dataset"], row["Method"])]
    observed = (round(row["Coverage"], 1), round(row["False-accept risk"], 1), row["FA"])
    assert observed == expected, (row["Dataset"], row["Method"], observed, expected)


METHOD_STYLES = {
    "raw-LLM": {
        "marker": "X",
        "facecolor": "#59636F",
        "edgecolor": "#374151",
        "size": 48,
    },
    "LLM-judge": {
        "marker": "^",
        "facecolor": "white",
        "edgecolor": "#59636F",
        "size": 48,
    },
    "judge+formula": {
        "marker": "s",
        "facecolor": "#D8DDE3",
        "edgecolor": "#59636F",
        "size": 45,
    },
    "PoT": {
        "marker": "P",
        "facecolor": "#D89B2B",
        "edgecolor": "#855E16",
        "size": 55,
    },
    "VeriFin": {
        "marker": "D",
        "facecolor": "#28618F",
        "edgecolor": "#173F60",
        "size": 68,
    },
}
INK, MUTED, GRID = "#17212B", "#5B6573", "#D7DCE2"


def _plot_risk_coverage(points, output_stem):
    with plt.rc_context(
        {
            "font.family": "serif",
            "font.serif": [
                "Times New Roman",
                "Times",
                "Liberation Serif",
                "DejaVu Serif",
            ],
            "font.size": 7.5,
            "axes.linewidth": 0.65,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.dpi": 180,
            "savefig.dpi": 600,
        }
    ):
        fig, axes = plt.subplots(
            1,
            2,
            figsize=(7.08, 2.62),
            sharex=True,
            sharey=True,
        )

        for ax, dataset in zip(axes, ["XBRLBench", "FinanceBench"]):
            panel = points[points["Dataset"] == dataset].set_index("Method")
            for method in METHOD_ORDER:
                row = panel.loc[method]
                style = METHOD_STYLES[method]
                ax.scatter(
                    row["Coverage"],
                    row["False-accept risk"],
                    marker=style["marker"],
                    s=style["size"],
                    facecolor=style["facecolor"],
                    edgecolor=style["edgecolor"],
                    linewidth=0.9,
                    zorder=4 if method == "VeriFin" else 3,
                )

            # Label the strongest baseline and the proposed method; the legend
            # identifies the remaining operating points without crowding them.
            pot = panel.loc["PoT"]
            verifin = panel.loc["VeriFin"]
            if dataset == "XBRLBench":
                pot_offset = (-7, 14)
                pot_align = "right"
                vf_offset = (7, 5)
                vf_align = "left"
            else:
                pot_offset = (-7, 9)
                pot_align = "right"
                vf_offset = (7, 10)
                vf_align = "left"

            ax.annotate(
                f"PoT\nFA={int(pot['FA'])}/{int(pot['Accepted'])}",
                (pot["Coverage"], pot["False-accept risk"]),
                xytext=pot_offset,
                textcoords="offset points",
                ha=pot_align,
                va="bottom",
                fontsize=6.8,
                color="#855E16",
                zorder=5,
            )
            ax.annotate(
                f"VeriFin\nFA={int(verifin['FA'])}/{int(verifin['Accepted'])}",
                (verifin["Coverage"], verifin["False-accept risk"]),
                xytext=vf_offset,
                textcoords="offset points",
                ha=vf_align,
                va="bottom",
                fontsize=6.9,
                color="#173F60",
                fontweight="bold",
                zorder=5,
            )

            total = DATASET_TOTALS[dataset]
            ax.set_title(
                f"{dataset} ($n={total}$)",
                loc="left",
                fontsize=8.2,
                fontweight="bold",
                color=INK,
                pad=7,
            )
            ax.set_xlim(78, 101.3)
            ax.set_xticks([80, 85, 90, 95, 100])
            ax.set_ylim(-1.4, 31.5)
            ax.set_yticks([0, 5, 10, 15, 20, 25, 30])
            ax.grid(color=GRID, linewidth=0.45, alpha=0.85, zorder=1)
            ax.axhline(0, color="#8C96A3", linewidth=0.65, zorder=2)
            ax.set_xlabel(
                r"Decision coverage, $1-\mathrm{abstention}$ (\%)",
                fontsize=7.1,
                color=INK,
                labelpad=5,
            )
            ax.tick_params(
                axis="both",
                labelsize=6.8,
                colors=MUTED,
                width=0.5,
                length=2.5,
            )
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.spines["left"].set_color("#98A1AC")
            ax.spines["bottom"].set_color("#98A1AC")

        axes[0].set_ylabel(
            r"False-accept risk (\%)",
            fontsize=7.1,
            color=INK,
            labelpad=5,
        )

        handles = [
            Line2D(
                [0],
                [0],
                linestyle="none",
                marker=METHOD_STYLES[method]["marker"],
                markersize=(METHOD_STYLES[method]["size"] ** 0.5) * 0.72,
                markerfacecolor=METHOD_STYLES[method]["facecolor"],
                markeredgecolor=METHOD_STYLES[method]["edgecolor"],
                markeredgewidth=0.9,
                label=method,
            )
            for method in METHOD_ORDER
        ]
        fig.legend(
            handles=handles,
            labels=METHOD_ORDER,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.005),
            ncol=5,
            frameon=False,
            handletextpad=0.35,
            columnspacing=1.25,
            fontsize=7.0,
        )
        fig.subplots_adjust(
            left=0.085,
            right=0.993,
            bottom=0.245,
            top=0.79,
            wspace=0.16,
        )

        output_stem = Path(output_stem)
        output_stem.parent.mkdir(parents=True, exist_ok=True)
        export_options = {
            "bbox_inches": "tight",
            "pad_inches": 0.02,
            "facecolor": "white",
        }
        fig.savefig(f"{output_stem}.pdf", **export_options)
        fig.savefig(f"{output_stem}.svg", **export_options)
        fig.savefig(f"{output_stem}.png", dpi=600, **export_options)
        return fig


_risk_coverage_figure = _plot_risk_coverage(
    risk_coverage_points,
    RISK_COVERAGE_ROOT / "figures/risk_coverage_operating_points",
)
plt.show()


**Paper caption.** Risk–coverage operating points on XBRLBench and FinanceBench
(lower right is better). Decision coverage is the fraction of all benchmark
questions receiving a binary verification outcome; all 67 FinanceBench
questions are included, so the three questions without a parseable shared
one-pool candidate count as uncovered for every baseline. False-accept risk is
$\mathrm{FA}/(\mathrm{TA}+\mathrm{FA})$, the fraction of accepted answers that
are incorrect. Each marker is one fixed configuration and markers are
intentionally unconnected because no confidence threshold is swept.
\textsc{VeriFin} is the only method with zero observed false-accept risk on
both datasets.


## Figure 1 — Acceptance outcomes by verification method

This full-width result figure retains every benchmark question: $n=600$ for
XBRLBench and $n=67$ for FinanceBench. Missing numerical candidates remain in
the gray ``not admitted'' portion rather than being removed from the denominator.
Counts are recomputed directly from the stored result files. All methods use the same frozen Claude Haiku 4.5 raw-answer text for each
question. The baseline runner reparses that text, recovering numeric values in
7 XBRLFiling and 6 FinanceBench cases where the source VeriFin artifact stored a
null `claimed_value`; parsing is therefore part of each evaluated method.


In [ ]:
# Full-denominator ICAIF main-result figure (FinanceBench N=67).
import runpy

_acceptance_namespace = runpy.run_path(
    str(ROOT / "scripts" / "plot_acceptance_outcomes.py")
)
acceptance_outcomes, _acceptance_outcomes_figure = _acceptance_namespace[
    "main"
](ROOT)


![Acceptance outcomes across both benchmarks](../figures/acceptance_outcomes.png)

**Paper caption.** Acceptance outcomes by verification method. Each bar
partitions all benchmark questions into correctly accepted claims, incorrectly
accepted claims (false accepts), and claims not admitted (rejected, abstained,
or lacking a numerical candidate). VeriFin records zero observed false accepts
on both datasets, whereas every baseline admits incorrect claims; its correctly
accepted segment shows that this result is not obtained by rejecting every
answer. All 67 FinanceBench questions are retained.


## Retrieval vs. oracle evidence (FinanceBench, agent + det-SMT)

Same codepath (the agent, deterministic SMT) with only the evidence source changed: oracle annotation vs. real dense retrieval + cross-encoder reranking over the filing corpus. Isolates retrieval as the single variable. Soundness (FA) should be retrieval-independent; retrieval quality should move coverage/abstention.

In [ ]:
import json, re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])
def _rid(r): return r.get("id") or r.get("financebench_id")
def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out
def _ok(r):
    adjudicated = _adjudicated_ok(_rid(r), r.get("claimed_value"))
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    cv=r.get("claimed_value")
    if cv is None: return None
    t=_nums(r.get("gold_answer") or r.get("answer"))
    return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None

# Answerable = FinanceBench questions with a parseable numeric candidate (from the raw
# baseline). Abstention is verifier abstention over these; candidate-generation failures
# are excluded, consistent with the main-results convention.
def _parseable(d):
    return {_rid(json.loads(l)) for l in (ROOT/"results"/d/"results.jsonl").read_text().splitlines()
            if l.strip() and json.loads(l).get("claimed_value") is not None}
PARSEABLE = _parseable("onepool/fb_none")

def _score(d):
    TA=FA=TC=FR=0; status={}
    for line in (ROOT/"results"/d/"results.jsonl").read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line); status[_rid(r)]=r.get("status"); cv=r.get("claimed_value")
        if cv is None: continue
        o=_ok(r)
        if r.get("status")=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif r.get("status")=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)
    ver=TA+FA; dec=ver+TC+FR
    nd=sum(1 for i in PARSEABLE if status.get(i) not in ("VERIFIED","VIOLATED"))
    return {"Precision":round(100*TA/ver,1) if ver else float("nan"),
            "Accuracy":round(100*(TA+TC)/dec,1) if dec else float("nan"),
            "Abstention":round(100*nd/len(PARSEABLE),1),"FA":FA}

rows=[{"Evidence":"Oracle", **_score("fb_detsmt_v2_haiku")},
      {"Evidence":"Retrieval (RAG)", **_score("fb_detsmt_v2_haiku_agent_rag")}]
retrieval_vs_oracle = pd.DataFrame(rows)[["Evidence","Precision","Accuracy","Abstention","FA"]]
print(f"answerable (parseable-candidate) questions: {len(PARSEABLE)}/67")
print(retrieval_vs_oracle.to_string(index=False))
print("\nSame agent + deterministic SMT; only evidence source differs.")
print("Soundness holds under real retrieval (FA=0); coverage is the cost (abstention rises).")
retrieval_vs_oracle


## Soundness: pooled false-accept bound

The zero-false-accept claim, framed statistically. We pool the negatives (incorrect answers the 
verifier adjudicated) across all four answer models and both datasets, and report the exact 
Clopper--Pearson upper bound on the false-accept rate. FinanceBench excludes the 5 non-numeric 
questions.

In [ ]:
import json, re
from pathlib import Path
from scipy.stats import beta
ROOT=Path.cwd()
if not (ROOT/"results").exists() and (ROOT.parent/"results").exists(): ROOT=ROOT.parent
ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])
NON_NUMERIC = set()  # all 67 questions remain in scope
def _rid(r): return r.get("id") or r.get("financebench_id")
def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out
def _ok(r):
    adjudicated = _adjudicated_ok(_rid(r), r.get("claimed_value"))
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    if r.get("claimed_value") is None: return None
    t=_nums(r.get("gold_answer") or r.get("answer"))
    return any(abs(abs(r["claimed_value"])-x)<=max(1.0,x*0.01) for x in t) if t else None
def negatives(d, excl):
    FA=TC=0
    for l in (ROOT/"results"/d/"results.jsonl").read_text().splitlines():
        if not l.strip(): continue
        r=json.loads(l)
        if _rid(r) in excl or r.get("claimed_value") is None: continue
        o=_ok(r); s=r.get("status")
        if s=="VERIFIED" and o is False: FA+=1
        elif s=="VIOLATED" and o is False: TC+=1
    return FA, FA+TC   # false accepts, and total incorrect answers adjudicated

RUNS={"XBRLBench":[("mc_calc_detsmt_v2_llama3.1-8b",set()),("mc_calc_detsmt_v2_qwen2.5-7b",set()),
                   ("mc_calc_detsmt_v2_haiku",set()),("latest/mc-qwen-30b-latest",set())],
      "FinanceBench":[("fb_detsmt_v2_llama3.1-8b",NON_NUMERIC),("fb_detsmt_qwen2.5-7b",NON_NUMERIC),
                      ("fb_detsmt_v2_haiku",NON_NUMERIC),("latest/fb_qwen3-30b_updated2_det_smt_run3_new",NON_NUMERIC)]}
def cp_upper(fa,n,c=0.95):
    return 100*beta.ppf(1-(1-c)/2, fa+1, n-fa) if n else float("nan")
rows=[]; poolFA=poolN=0
for ds,runs in RUNS.items():
    FA=N=0
    for d,excl in runs:
        f,n=negatives(d,excl); FA+=f; N+=n
    poolFA+=FA; poolN+=N
    rows.append((ds,N,FA,round(cp_upper(FA,N),2),round(cp_upper(FA,N,0.99),2)))
rows.append(("Pooled (all 8 runs)",poolN,poolFA,round(cp_upper(poolFA,poolN),2),round(cp_upper(poolFA,poolN,0.99),2)))
import pandas as pd
fa_bound=pd.DataFrame(rows, columns=["Scope","Negatives (incorrect adjudicated)","False accepts","95% CP upper (%)","99% CP upper (%)"])
print(fa_bound.to_string(index=False))
print(f"\nAcross {poolN} incorrect answers pooled over 4 models x 2 datasets, VeriFin accepted {poolFA};")
print(f"exact Clopper-Pearson 95% upper bound on the false-accept rate = {cp_upper(poolFA,poolN):.2f}%.")
fa_bound


## Scoring at the filing's precision (false-reject audit)

The default scorer credits a candidate within $1\%$ of the gold. VeriFin instead enforces the 
filing's reporting precision, so near-misses it correctly rejects were being counted as false 
rejects. Here we re-score correctness at the filing's precision (unit-aware: money $\to$ 
$\max(0.5,0.1\%)$; percent/ratio $\to 0.05$), matching VeriFin's tolerance. This removes the 
false-reject artifacts, preserves VeriFin's FA$=0$, and shows the baselines accept more near-miss 
errors than the loose scorer credited.

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"results").exists() and (ROOT.parent/"results").exists(): ROOT=ROOT.parent
ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])
def _rid(r): return r.get("id") or r.get("financebench_id")
def _gtok(g):
    out=[]
    for tok in re.findall(r"-?\d[\d,]*\.?\d*", str(g or "")):
        t=tok.replace(",","")
        try: v=abs(float(t))
        except ValueError: continue
        out.append((v, len(t.split(".",1)[1]) if "." in t else 0))
    return out
def ok_loose(r):
    adjudicated = _adjudicated_ok(_rid(r), r.get("claimed_value"))
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    cv=r.get("claimed_value")
    if cv is None: return None
    t=[v for v,_ in _gtok(r.get("gold_answer") or r.get("answer"))]
    return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None
_MONEY_KW=("income","revenue","assets","liabilit","cash","expense","capital","equity","profit",
           "ppe","payable","receivable","ebitda","dividend","capex","inventory","proceeds","gain",
           "charges","working")
def _unit_of(gold, metric):
    # Derive the reporting unit from the gold text markers and the metric name (not the value),
    # so the tolerance matches VeriFin's tau: percent 0.05, ratio 0.005, money max(0.5, 0.1%).
    g=str(gold).lower(); m=str(metric or "").lower()
    if "%" in g or any(k in m for k in ("percent","margin","_rate","change","growth")): return "percent"
    if "ratio" in m or "turnover" in m: return "ratio"
    if "$" in g or "million" in g or "billion" in g or "thousand" in g or any(k in m for k in _MONEY_KW): return "money"
    return "number"
def _tol_for(v,d,unit):
    if unit=="percent": return max(0.05, 0.5*10**(-d))
    if unit=="ratio":   return max(0.005, 0.5*10**(-d))
    if unit=="money":   return max(0.5, 0.001*v)
    return 0.5*10**(-d)
def ok_filing(r):
    adjudicated = _adjudicated_ok(_rid(r), r.get("claimed_value"))
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    cv=r.get("claimed_value")
    if cv is None: return None
    g=r.get("gold_answer") or r.get("answer"); u=_unit_of(g, r.get("metric"))
    for v,d in _gtok(g):
        if abs(abs(cv)-v)<=_tol_for(v,d,u): return True
    return False
def _counts(d,okf):
    TA=FA=TC=FR=0
    for l in (ROOT/"results"/d/"results.jsonl").read_text().splitlines():
        if not l.strip(): continue
        r=json.loads(l); s=r.get("status"); cv=r.get("claimed_value")
        if cv is None: continue
        o=okf(r)
        if s=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif s=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)
    return TA,FA,TC,FR
DS={"XBRLBench":[("raw-LLM","onepool/xbrl_none"),("LLM-judge","onepool/xbrl_judge"),
     ("judge+formula","onepool/xbrl_judge_formula"),("PoT","onepool/xbrl_pot"),("VeriFin","mc_calc_detsmt_v2_haiku")],
    "FinanceBench":[("raw-LLM","onepool/fb_none"),("LLM-judge","onepool/fb_judge"),
     ("judge+formula","onepool/fb_judge_formula"),("PoT","onepool/fb_pot"),("VeriFin","fb_detsmt_v2_haiku")]}
rows=[]
for ds,runs in DS.items():
    for name,d in runs:
        _,FAl,_,FRl=_counts(d,ok_loose)
        TA,FA,TC,FR=_counts(d,ok_filing)
        corr=TA+FR
        rows.append({"dataset":ds,"method":name,"FA (1%)":FAl,"FA (filing)":FA,
                     "FR (1%)":FRl,"FR (filing)":FR,"FR-rate% (filing)":round(100*FR/corr,1) if corr else 0.0,
                     "Precision% (filing)":round(100*TA/(TA+FA),1) if TA+FA else 100.0})
filing_precision_rescore = pd.DataFrame(rows)
print(filing_precision_rescore.to_string(index=False))
print("\nVeriFin retains FA=0 and 100% precision under filing-precision scoring on both datasets;")
print("its false rejects fall (the loose-scorer artifacts vanish), while the baselines' false")
print("accepts rise (they were credited for accepting near-misses the filing's precision rejects).")
filing_precision_rescore


## Main results at filing precision (corrected scorer)

The main table re-scored at the filing's reporting precision (unit-aware, matching VeriFin's 
tolerance) instead of the loose $1\%$ default. VeriFin's FA$=0$ and $100\%$ precision are 
preserved; its accuracy rises because near-miss rejections are now correctly credited, while the 
baselines' false accepts rise because accepting a near-miss is no longer scored as correct.

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"results").exists() and (ROOT.parent/"results").exists(): ROOT=ROOT.parent
ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])
def _rid(r): return r.get("id") or r.get("financebench_id")
def _gtok(g):
    out=[]
    for tok in re.findall(r"-?\d[\d,]*\.?\d*", str(g or "")):
        t=tok.replace(",","")
        try: v=abs(float(t))
        except ValueError: continue
        out.append((v,len(t.split(".",1)[1]) if "." in t else 0))
    return out
_MONEY_KW=("income","revenue","assets","liabilit","cash","expense","capital","equity","profit",
           "ppe","payable","receivable","ebitda","dividend","capex","inventory","proceeds","gain",
           "charges","working")
def _unit_of(gold, metric):
    # Derive the reporting unit from the gold text markers and the metric name (not the value),
    # so the tolerance matches VeriFin's tau: percent 0.05, ratio 0.005, money max(0.5, 0.1%).
    g=str(gold).lower(); m=str(metric or "").lower()
    if "%" in g or any(k in m for k in ("percent","margin","_rate","change","growth")): return "percent"
    if "ratio" in m or "turnover" in m: return "ratio"
    if "$" in g or "million" in g or "billion" in g or "thousand" in g or any(k in m for k in _MONEY_KW): return "money"
    return "number"
def _tol_for(v,d,unit):
    if unit=="percent": return max(0.05, 0.5*10**(-d))
    if unit=="ratio":   return max(0.005, 0.5*10**(-d))
    if unit=="money":   return max(0.5, 0.001*v)
    return 0.5*10**(-d)
def ok_filing(r):
    adjudicated = _adjudicated_ok(_rid(r), r.get("claimed_value"))
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    cv=r.get("claimed_value")
    if cv is None: return None
    g=r.get("gold_answer") or r.get("answer"); u=_unit_of(g, r.get("metric"))
    for v,d in _gtok(g):
        if abs(abs(cv)-v)<=_tol_for(v,d,u): return True
    return False
def _par(d): return {_rid(r) for r in (json.loads(l) for l in (ROOT/"results"/d/"results.jsonl").read_text().splitlines() if l.strip()) if r.get("claimed_value") is not None}
def _score(d,par):
    TA=FA=TC=FR=0; st={}
    for l in (ROOT/"results"/d/"results.jsonl").read_text().splitlines():
        if not l.strip(): continue
        r=json.loads(l); st[_rid(r)]=r.get("status"); cv=r.get("claimed_value")
        if cv is None: continue
        o=ok_filing(r)
        if r.get("status")=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif r.get("status")=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)
    ver=TA+FA; dec=ver+TC+FR
    nd=sum(1 for i in par if st.get(i) not in ("VERIFIED","VIOLATED"))
    return {"Precision":100*TA/ver if ver else float("nan"),"Accuracy":100*(TA+TC)/dec if dec else float("nan"),
            "Abstention":100*nd/len(par) if par else float("nan"),"FA":FA}
DS=[("XBRLBench","onepool/xbrl_none",{"raw-LLM":"onepool/xbrl_none","LLM-judge":"onepool/xbrl_judge",
     "judge+formula":"onepool/xbrl_judge_formula","PoT":"onepool/xbrl_pot","VeriFin (ours)":"mc_calc_detsmt_v2_haiku"}),
    ("FinanceBench","onepool/fb_none",{"raw-LLM":"onepool/fb_none","LLM-judge":"onepool/fb_judge",
     "judge+formula":"onepool/fb_judge_formula","PoT":"onepool/fb_pot","VeriFin (ours)":"fb_detsmt_v2_haiku"})]
METHODS=["raw-LLM","LLM-judge","judge+formula","PoT","VeriFin (ours)"]
cols={}
for ds,base,runs in DS:
    par=_par(base)
    for m in METHODS:
        s=_score(runs[m],par)
        for k,v in s.items(): cols.setdefault((ds,k),{})[m]=v
main_results_filing = pd.DataFrame(cols).reindex(METHODS)
main_results_filing.columns=pd.MultiIndex.from_tuples(main_results_filing.columns)
def _f(v,c): return f"{int(v)}" if c[1]=="FA" else f"{v:.1f}"
print(pd.DataFrame({c:[_f(v,c) for v in main_results_filing[c]] for c in main_results_filing.columns}, index=METHODS).to_string())
main_results_filing


## Main-results bar chart — publication-scale side-by-side panels

This version preserves the supplied grouped-bar design while drawing at the
ICAIF full-text width (7.08 in). It reads `main_results_filing` directly so the
table and figure cannot drift. Precision is $\mathrm{TA}/(\mathrm{TA}+\mathrm{FA})$;
decision accuracy is computed over numeric candidates receiving a binary
verdict; abstention is a non-decision among baseline-parseable candidates; and
FA annotations are counts. Panel $n$ labels give the full dataset sizes.


In [ ]:
"""Publication-scale VeriFin main-results bar chart."""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Patch


METHOD_LABELS = ["Direct\nLLM", "LLM\njudge", "Judge +\nformula", "PoT", "VeriFin"]
RESULT_METHODS = [
    "raw-LLM", "LLM-judge", "judge+formula", "PoT", "VeriFin (ours)"
]
METRICS = ("Prec", "Acc", "Abst")
METRIC_COLUMNS = {
    "Prec": "Precision", "Acc": "Accuracy", "Abst": "Abstention"
}
COLORS = {"Prec": "#5bb6a9", "Acc": "#33566e", "Abst": "#cbb9d6"}
METRIC_LABELS = {
    "Prec": "Precision", "Acc": "Decision accuracy", "Abst": "Abstention"
}
HIGHLIGHT = "#f6e2b3"
HIGHLIGHT_EDGE = "#b7791f"


if "main_results" not in globals():
    raise RuntimeError(
        "Run the preceding 'Main results' cell first."
    )


def _repo_root():
    for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/verifiqa")):
        candidate = candidate.resolve()
        if (candidate / "notebooks/agent_financebench_result_metrics.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory.")


def _panel(dataset):
    data = {}
    for key, column in METRIC_COLUMNS.items():
        data[key] = [
            round(float(main_results.loc[method, (dataset, column)]), 1)
            for method in RESULT_METHODS
        ]
    data["FA"] = [
        int(round(float(main_results.loc[method, (dataset, "FA")])))
        for method in RESULT_METHODS
    ]
    return data


def _add_value(ax, x_pos, value, metric):
    label = f"{value:g}"
    if value == 0:
        ax.annotate(
            label, (x_pos, 0), xytext=(0, 2.0), textcoords="offset points",
            ha="center", va="bottom", rotation=90, fontsize=7.0,
            fontweight="semibold", color="#8a8a8a", clip_on=True, zorder=5,
        )
        return

    # Tall bars carry their labels inside the bar; short bars place them above.
    if value >= 24:
        color = "white" if metric == "Acc" else "#111111"
        ax.annotate(
            label, (x_pos, value), xytext=(0, -2.2), textcoords="offset points",
            ha="center", va="top", rotation=90, fontsize=7.5,
            fontweight="bold", color=color, clip_on=True, zorder=5,
        )
    else:
        ax.annotate(
            label, (x_pos, value), xytext=(0, 2.0), textcoords="offset points",
            ha="center", va="bottom", rotation=90, fontsize=7.5,
            fontweight="bold", color="#111111", clip_on=True, zorder=5,
        )


def _plot_panel(ax, data, title, show_y):
    centers = np.arange(len(METHOD_LABELS), dtype=float)
    bar_width = 0.205

    ax.axvspan(3.53, 4.47, color=HIGHLIGHT, zorder=0)
    for metric_index, metric in enumerate(METRICS):
        positions = centers + (metric_index - 1) * bar_width
        bars = ax.bar(
            positions,
            data[metric],
            width=bar_width,
            color=COLORS[metric],
            edgecolor="none",
            linewidth=0,
            zorder=3,
        )
        for position, value in zip(positions, data[metric]):
            _add_value(ax, position, value, metric)

    # FA is a count, not a percentage: place it in a dedicated row above the axis.
    for index, (center, count) in enumerate(zip(centers, data["FA"])):
        ax.text(
            center,
            1.055,
            f"FA {count}",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            fontsize=7.5,
            fontweight="bold",
            color=HIGHLIGHT_EDGE if index == len(centers) - 1 else "#5f5f5f",
            clip_on=False,
        )

    ax.text(
        0.5, 1.255, title, transform=ax.transAxes, ha="center", va="bottom",
        fontsize=9.0, fontweight="bold", clip_on=False,
    )
    ax.set_xlim(-0.48, 4.48)
    ax.set_ylim(0, 105)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.set_xticks(centers)
    ax.set_xticklabels(METHOD_LABELS, fontsize=7.5)
    ax.tick_params(axis="x", length=0, pad=3)
    ax.tick_params(axis="y", labelsize=7.5, width=0.6, length=2.5)
    if show_y:
        ax.set_ylabel("Rate (%)", fontsize=8.5, labelpad=3)
    else:
        ax.tick_params(axis="y", labelleft=False, left=False)
        ax.spines["left"].set_visible(False)

    ax.grid(axis="y", color="#d7d7d7", linewidth=0.45, alpha=0.72, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.65)
    if show_y:
        ax.spines["left"].set_linewidth(0.65)


root = _repo_root()
output_stem = root / "figures" / "verifin_results"
output_stem.parent.mkdir(parents=True, exist_ok=True)

with plt.rc_context({
    "font.family": "serif",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}):
    fig, axes = plt.subplots(1, 2, figsize=(7.08, 3.05), sharey=True)
    _plot_panel(axes[0], _panel("XBRLBench"), "XBRLFiling", True)
    _plot_panel(axes[1], _panel("FinanceBench"), "FinanceBench", False)

    legend = [
        Patch(
            facecolor=COLORS[metric], edgecolor="none", linewidth=0,
            label=METRIC_LABELS[metric],
        )
        for metric in METRICS
    ]
    legend.append(
        Patch(
            facecolor=HIGHLIGHT, edgecolor="none", linewidth=0,
            label="VeriFin (ours)",
        )
    )
    fig.legend(
        handles=legend, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 0.985),
        frameon=False, fontsize=8.0, handlelength=1.25, handleheight=0.85,
        handletextpad=0.45, columnspacing=1.2,
    )
    fig.subplots_adjust(
        left=0.078, right=0.995, bottom=0.205, top=0.67, wspace=0.10
    )
    fig.savefig(output_stem.with_suffix(".pdf"), facecolor="white")
    fig.savefig(output_stem.with_suffix(".png"), dpi=600, facecolor="white")
    print("SAVED:", output_stem, "(7.08 x 3.05 in; vector PDF + 600-dpi PNG)")
    display(fig)
    plt.close(fig)


In [ ]:
import json, re
from pathlib import Path
import pandas as pd
nan=float("nan")

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])

def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out

def _rid(r): return r.get("id") or r.get("financebench_id")

def _ok(rid, cv, g, kc):
    adjudicated = _adjudicated_ok(rid, cv) if kc else _NO_ADJUDICATION
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    if cv is None: return None
    t=_nums(g)
    return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None

def _parseable_ids(path):
    """Answerable questions: those with a parseable numeric candidate, measured from the
    raw-LLM baseline (which parses every candidate). Questions with no numeric candidate
    are candidate-generation failures, shared by all methods and excluded from abstention."""
    ids=set()
    for line in Path(path).read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line)
        if r.get("claimed_value") is not None: ids.add(_rid(r))
    return ids

def _score(path, kc, parseable):
    TA=TC=FR=FA=0; status={}
    for line in Path(path).read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line); rid=_rid(r); st=r.get("status"); cv=r.get("claimed_value")
        status[rid]=st
        if cv is None: continue                      # no candidate: not a decision
        o=_ok(rid, cv, r.get("gold_answer") or r.get("answer"), kc)
        if st=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif st=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)
    ver=TA+FA; dec=ver+TC+FR
    # Abstention = AB/n over all benchmark questions, matching the paper's metric
    # definition. Questions with no parseable candidate count as declined.
    nd_all=sum(1 for st in status.values() if st not in ("VERIFIED","VIOLATED"))
    return {"Precision": 100*TA/ver if ver else nan,
            "Accuracy": 100*(TA+TC)/dec if dec else nan,
            "Abstention": 100*nd_all/len(status) if status else nan,
            "FA": FA}

DATASETS = [
    ("XBRLBench", False, {
        "raw-LLM":"onepool/xbrl_none", "LLM-judge":"onepool/xbrl_judge",
        "judge+formula":"onepool/xbrl_judge_formula", "PoT":"onepool/xbrl_pot",
        "VeriFin (ours)":"mc_calc_detsmt_v2_haiku"}),
    ("FinanceBench", True, {
        "raw-LLM":"onepool/fb_none", "LLM-judge":"onepool/fb_judge",
        "judge+formula":"onepool/fb_judge_formula", "PoT":"onepool/fb_pot",
        "VeriFin (ours)":"fb_detsmt_v2_haiku"}),
]
METHODS = ["raw-LLM","LLM-judge","judge+formula","PoT","VeriFin (ours)"]

cols={}
answerable={}
for dset, kc, runs in DATASETS:
    parseable=_parseable_ids(ROOT/"results"/runs["raw-LLM"]/"results.jsonl")
    answerable[dset]=len(parseable)
    for m in METHODS:
        s=_score(ROOT/"results"/runs[m]/"results.jsonl", kc, parseable)
        for metric,val in s.items():
            cols.setdefault((dset,metric),{})[m]=val

main_results = pd.DataFrame(cols).reindex(METHODS)
main_results.columns = pd.MultiIndex.from_tuples(main_results.columns)
print("answerable (parseable-candidate) questions:", answerable)
def _fmt(v,c): return f"{int(v)}" if c[1]=="FA" else f"{v:.1f}"
print(pd.DataFrame({c:[_fmt(v,c) for v in main_results[c]] for c in main_results.columns},
                   index=METHODS).to_string())

# ---- paper LaTeX (tab:main), emitted from the same numbers ----
def _cell(m, dset, metric, bold_if_best=False):
    v=main_results.loc[m,(dset,metric)]
    s=f"{int(v)}" if metric=="FA" else f"{v:.1f}"
    return s
best={("XBRLBench","Precision"),("XBRLBench","Accuracy"),("XBRLBench","FA"),
      ("FinanceBench","Precision"),("FinanceBench","FA")}   # cells VeriFin wins
lines=[]
for m in METHODS:
    cells_=[]
    for dset in ["XBRLBench","FinanceBench"]:
        for metric in ["Precision","Accuracy","Abstention","FA"]:
            s=_cell(m,dset,metric)
            if m=="VeriFin (ours)" and (dset,metric) in best: s=f"\\bfseries {s}"
            cells_.append(s)
    row="  & "+" & ".join(cells_)+" \\\\"
    if m=="VeriFin (ours)": row="\\rowcolor{ourshl}\n"+m+row
    else: row=m+row
    lines.append(row)
print("\n% ---- tab:main LaTeX (auditable, from result files) ----")
print("\n".join(lines))
main_results


![VeriFin main-results bar chart](../figures/verifin_results.png)

## Figure: risk--recall operating points (ROC)

Each method as a point in ROC space (false-accept rate vs recall), per dataset, scored from each 
run's own file. VeriFin sits on the FPR=0 axis (the audit-safe extreme) at high recall; baselines 
trade false accepts for coverage. Saves `figures/roc.{pdf,png}`.

In [ ]:
import json, re
from pathlib import Path
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/"results").exists() and (ROOT.parent/"results").exists(): ROOT=ROOT.parent
ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])
NON_NUMERIC = set()  # all 67 questions remain in scope
def _rid(r): return r.get("id") or r.get("financebench_id")
def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out
def _ok(r):
    adjudicated = _adjudicated_ok(_rid(r), r.get("claimed_value"))
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    if r.get("claimed_value") is None: return None
    t=_nums(r.get("gold_answer") or r.get("answer"))
    return any(abs(abs(r["claimed_value"])-x)<=max(1.0,x*0.01) for x in t) if t else None
def roc(d, excl):
    TA=FA=TC=FR=0
    for l in (ROOT/"results"/d/"results.jsonl").read_text().splitlines():
        if not l.strip(): continue
        r=json.loads(l)
        if _rid(r) in excl or r.get("claimed_value") is None: continue
        o=_ok(r); s=r.get("status")
        if s=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif s=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)
    return (FA/(FA+TC) if FA+TC else 0.0, TA/(TA+FR) if TA+FR else 0.0)
DS={"XBRLBench":([("Raw-LLM","onepool/xbrl_none"),("LLM-judge","onepool/xbrl_judge"),
     ("Judge+formula","onepool/xbrl_judge_formula"),("PoT","onepool/xbrl_pot"),("VeriFin","mc_calc_detsmt_v2_haiku")], set()),
    "FinanceBench":([("Raw-LLM","onepool/fb_none"),("LLM-judge","onepool/fb_judge"),
     ("Judge+formula","onepool/fb_judge_formula"),("PoT","onepool/fb_pot"),("VeriFin","fb_detsmt_v2_haiku")], NON_NUMERIC)}
NEUTRAL="#6B7280"; OURS="#0F766E"; INK="#374151"
plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],"font.size":11,"axes.linewidth":0.8})
fig,axes=plt.subplots(1,2,figsize=(10.6,4.6),gridspec_kw={"wspace":0.14})
OFF={"Raw-LLM":(-6,-14),"LLM-judge":(8,-4),"Judge+formula":(8,-2),"PoT":(9,3),"VeriFin":(0,17)}
for ax,(ds,(rows,excl)) in zip(axes,DS.items()):
    ax.axvspan(-0.02,0.02,color=OURS,alpha=0.08,zorder=0)
    ax.plot([0,1],[0,1],"--",color="#C9CDD3",lw=1.2,zorder=1)
    for m,d in rows:
        fpr,tpr=roc(d,excl); ours=m=="VeriFin"
        ax.scatter([fpr],[tpr], s=180 if ours else 90, marker="*" if ours else "o",
                   color=OURS if ours else NEUTRAL, edgecolor="white", linewidth=1.1, zorder=5)
        dx,dy=OFF[m]
        ax.annotate(m,(fpr,tpr),xytext=(dx,dy),textcoords="offset points",fontsize=10,
                    color=OURS if ours else INK, fontweight="bold" if ours else "normal",
                    ha="center" if ours else ("left" if dx>0 else "right"), va="center")
    ax.set_xlim(-0.05,1.05); ax.set_ylim(0,1.05)
    ax.set_xlabel("False-accept rate  (FPR $\\rightarrow$ risk)", fontsize=11)
    if ax is axes[0]: ax.set_ylabel("Recall  (correct answers accepted)", fontsize=11)
    ax.set_title(ds, fontsize=12.5, fontweight="bold", loc="left", pad=8)
    ax.grid(True, lw=0.4, alpha=0.3); ax.set_axisbelow(True); ax.tick_params(length=0)
    ax.spines[["top","right"]].set_visible(False)
    ax.spines["left"].set_color("#9CA3AF"); ax.spines["bottom"].set_color("#9CA3AF")
    ax.annotate("audit-safe\n(no false accepts)", (0.0,0.30), xytext=(0.10,0.20), fontsize=8.5,
                color=OURS, ha="left", arrowprops=dict(arrowstyle="->",color=OURS,lw=0.8))
fig.suptitle("VeriFin operates at the conservative extreme: zero false-accept risk at high recall",
             fontsize=12.8, fontweight="bold", x=0.02, ha="left", y=1.01)
fig.savefig(ROOT/"figures"/"roc.pdf", bbox_inches="tight")
fig.savefig(ROOT/"figures"/"roc.png", dpi=200, bbox_inches="tight")
plt.show()


## Supporting figure — Precision and abstention

Precision bars with exact Clopper--Pearson 95% intervals; abstention beside each panel. Saves `figures/three_metrics.{pdf,png}`.

In [ ]:
import json, re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out
def _ok(cv,g):
    if cv is None: return None
    t=_nums(g)
    return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None
def _counts(path):
    TA=TC=FR=FA=AB=n=0
    for line in Path(path).read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line); n+=1
        st=r.get("status"); cv=r.get("claimed_value")
        o=_ok(cv, r.get("gold_answer") or r.get("answer"))
        if st in ("ABSTAIN","UNVERIFIED_FORMULA") or cv is None: AB+=1
        elif st=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif st=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)  # non-numeric gold + numeric candidate rejected = true reject
        else: AB+=1
    return dict(n=n,TA=TA,TC=TC,FR=FR,FA=FA,AB=AB)

DET = [("Llama-3.1-8B","mc_calc_detsmt_v2_llama3.1-8b"),
       ("Qwen2.5-7B","mc_calc_detsmt_v2_qwen2.5-7b"),
       ("Claude Haiku 4.5","mc_calc_detsmt_v2_haiku"),
       ("Qwen3-30B","latest/mc-qwen-30b-latest"),
       ("Fino1","latest/mc-fino-30b-latest")]
rows=[]
for name,d in DET:
    c=_counts(ROOT/"results"/d/"results.jsonl")
    ver=c["TA"]+c["FA"]; dec=ver+c["TC"]+c["FR"]
    rows.append({"Answer model":name,
                 "Prec.":round(100*c["TA"]/ver,1) if ver else float("nan"),
                 "Catch":round(100*c["TC"]/(c["TC"]+c["FA"]),1) if (c["TC"]+c["FA"]) else 100.0,
                 "FA":c["FA"], "FR":c["FR"],
                 "Abst.":round(100*c["AB"]/c["n"],1)})
models_detsmt = pd.DataFrame(rows)
print(models_detsmt.to_string(index=False))
models_detsmt


In [ ]:

# Main-paper figure: upright XBRLBench rejection bars only.
# Optional repair side-channel outputs are ignored so deterministic SMT stays fixed.
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

paper_model_runs = [
    ("GPT-5.5", ROOT / "results/latest/mc_calc_gpt5.5_codex_det_smt_Nrepair_run1/results.jsonl"),
    ("Claude Haiku 4.5", ROOT / "results/mc_calc_detsmt_v2_haiku/results.jsonl"),
    ("Fino1", ROOT / "results/latest/mc-fino-30b-latest/results.jsonl"),
    ("Qwen3-30B", ROOT / "results/latest/mc-qwen-30b-latest/results.jsonl"),
    ("Llama-3.1-8B", ROOT / "results/mc_calc_detsmt_v2_llama3.1-8b/results.jsonl"),
    ("Qwen2.5-7B", ROOT / "results/mc_calc_detsmt_v2_qwen2.5-7b/results.jsonl"),
]
MODEL_ORDER = [model for model, _ in paper_model_runs]

xbrl_question_by_id = {
    row["financebench_id"]: row
    for row in (
        json.loads(line)
        for line in (
            ROOT / "data/multicompany_provable/data/provable_calcrequired.jsonl"
        ).read_text().splitlines()
        if line.strip()
    )
}

FAMILY_TERMS = [
    ("gross profit", "Income statement"),
    ("operating income", "Income statement"),
    ("operating expenses", "Income statement"),
    ("income before income taxes", "Income statement"),
    ("net income", "Income statement"),
    ("total current assets", "Balance sheet"),
    ("total non-current assets", "Balance sheet"),
    ("total current liabilities", "Balance sheet"),
    ("total non-current liabilities", "Balance sheet"),
    ("total liabilities", "Balance sheet"),
    ("total shareholders", "Balance sheet"),
    ("net property", "Balance sheet"),
    ("net cash from operating", "Cash flow"),
    ("net cash from investing", "Cash flow"),
    ("net cash from financing", "Cash flow"),
]
FAMILY_ORDER = ["Income statement", "Balance sheet", "Cash flow"]


def _statement_family(question):
    lowered = question.lower()
    for term, family in FAMILY_TERMS:
        if term in lowered:
            return family
    raise ValueError(f"Unmapped XBRLBench question: {question}")


def _paper_numbers(text):
    values = []
    for token in re.findall(r"-?\d[\d,]*\.?\d*", str(text or "")):
        try:
            values.append(abs(float(token.replace(",", ""))))
        except ValueError:
            pass
    return values


def _paper_correct(claimed_value, gold):
    """Paper convention: magnitude match within max($1m, 1%)."""
    if claimed_value is None:
        return False
    gold_values = _paper_numbers(gold)
    return any(
        abs(abs(float(claimed_value)) - value) <= max(1.0, value * 0.01)
        for value in gold_values
    )


def _xbrl_caught_by_family(path):
    caught = {family: 0 for family in FAMILY_ORDER}
    incorrect_numeric = {family: 0 for family in FAMILY_ORDER}
    false_accepts = 0
    unresolved_incorrect = 0
    rows = [
        json.loads(line)
        for line in Path(path).read_text().splitlines()
        if line.strip()
    ]
    assert len(rows) == 600
    assert len({row["id"] for row in rows}) == 600
    assert all(row["id"] in xbrl_question_by_id for row in rows)
    for row in rows:
        claimed_value = row.get("claimed_value")
        if claimed_value is None or _paper_correct(
            claimed_value,
            row.get("gold_answer") or row.get("answer"),
        ):
            continue
        family = _statement_family(xbrl_question_by_id[row["id"]]["question"])
        incorrect_numeric[family] += 1
        if row.get("status") == "VIOLATED":
            caught[family] += 1
        elif row.get("status") == "VERIFIED":
            false_accepts += 1
        else:
            unresolved_incorrect += 1
    return caught, incorrect_numeric, false_accepts, unresolved_incorrect


xbrl_caught_rows = []
xbrl_incorrect_rows = []
xbrl_false_accepts = {}
xbrl_unresolved = {}
for model, path in paper_model_runs:
    caught, incorrect, false_accepts, unresolved = _xbrl_caught_by_family(path)
    xbrl_caught_rows.append({"Answer model": model, **caught})
    xbrl_incorrect_rows.append({"Answer model": model, **incorrect})
    xbrl_false_accepts[model] = false_accepts
    xbrl_unresolved[model] = unresolved

xbrl_caught = pd.DataFrame(xbrl_caught_rows).set_index("Answer model")
xbrl_incorrect = pd.DataFrame(xbrl_incorrect_rows).set_index("Answer model")
assert xbrl_caught[FAMILY_ORDER].equals(xbrl_incorrect[FAMILY_ORDER])
assert all(value == 0 for value in xbrl_false_accepts.values())
assert all(value == 0 for value in xbrl_unresolved.values())
assert int(xbrl_caught[FAMILY_ORDER].to_numpy().sum()) == 1375

MODEL_TICK_LABELS = [
    "GPT-5.5",
    "Claude\nHaiku 4.5",
    "Fino1",
    "Qwen3\n30B",
    "Llama3.1\n8B",
    "Qwen2.5\n7B",
]
XBRL_SPECS = [
    ("Income statement", "Income statement", "#28618F", "#1D496C", ""),
    ("Balance sheet", "Balance sheet", "#2A8C82", "#1C625B", "///"),
    ("Cash flow", "Cash flow", "#D89B2B", "#8C631B", "..."),
]
INK, MUTED, GRID = "#17212B", "#5B6573", "#D4DAE1"

with plt.rc_context(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 9.0,
        "axes.linewidth": 0.65,
        "hatch.linewidth": 0.45,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
):
    fig, ax = plt.subplots(figsize=(4.00, 2.75))
    ordered = xbrl_caught.reindex(MODEL_ORDER)
    model_x = np.arange(len(MODEL_ORDER)) * 1.15
    bar_width = 0.33
    offsets = (-0.35, 0.0, 0.35)

    for (column, _, facecolor, edgecolor, hatch), offset in zip(
        XBRL_SPECS,
        offsets,
    ):
        values = ordered[column].to_numpy()
        bars = ax.bar(
            model_x + offset,
            values,
            width=bar_width,
            color=facecolor,
            edgecolor=edgecolor,
            linewidth=0.55,
            hatch=hatch,
            zorder=3,
        )
        for bar, value in zip(bars, values):
            ax.annotate(
                f"{int(value)}",
                (bar.get_x() + bar.get_width() / 2, value),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=6.8,
                color=INK,
                fontweight="bold",
                zorder=4,
            )

    ax.set_xlim(model_x[0] - 0.60, model_x[-1] + 0.60)
    ax.set_ylim(0, 268)
    ax.set_yticks([0, 50, 100, 150, 200, 250])
    ax.set_ylabel(
        "Wrong claims caught by VeriFin",
        fontsize=7.6,
        color=INK,
        labelpad=5,
    )
    ax.set_xticks(model_x, MODEL_TICK_LABELS, fontsize=7.4, color=INK)
    ax.grid(axis="y", color=GRID, linewidth=0.45, alpha=0.9, zorder=1)
    ax.tick_params(axis="x", length=0, pad=5)
    ax.tick_params(
        axis="y",
        length=2.3,
        width=0.55,
        colors=MUTED,
        labelsize=7.0,
    )
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#AEB5BE")

    legend_handles = [
        Patch(
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=0.55,
            hatch=hatch,
            label=label,
        )
        for _, label, facecolor, edgecolor, hatch in XBRL_SPECS
    ]
    ax.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.0, 1.01),
        ncol=3,
        frameon=False,
        fontsize=7.4,
        handlelength=1.05,
        handletextpad=0.4,
        columnspacing=0.8,
        borderaxespad=0,
    )

    fig.subplots_adjust(left=0.075, right=0.995, top=0.72, bottom=0.22)

    output_pdf = ROOT / "figures/category_model_accuracy_fixed_verifier.pdf"
    output_png = ROOT / "figures/category_model_accuracy_fixed_verifier.png"
    fig.savefig(output_pdf, bbox_inches="tight", pad_inches=0.02, facecolor="white")
    fig.savefig(
        output_png,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02,
        facecolor="white",
    )
    plt.show()


**Paper caption.** Incorrect numeric answers caught by VeriFin on XBRLBench
($n=600$ per answer model) with deterministic SMT held fixed. Bars show
caught-error counts by answer model and financial-statement family; each label
gives the count and its percentage of all 600 questions. Taller bars
indicate more upstream answer-model errors, not stronger verifier performance.
All counted wrong claims receive a violated verdict, with no observed false
accepts ($\mathrm{FA}=0$).


## Table 3 — Deterministic vs. autoformalized SMT (ablation)

Replacing deterministic compilation with an LLM that emits the SMT (autoformalization) keeps FA=0 but collapses coverage: the autoformalizer fails to produce a checkable formula on most questions. This isolates the value of deterministic compilation.

In [ ]:
import json, re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out
def _ok(cv,g):
    if cv is None: return None
    t=_nums(g)
    return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None
def _counts(path):
    TA=TC=FR=FA=AB=n=0
    for line in Path(path).read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line); n+=1
        st=r.get("status"); cv=r.get("claimed_value")
        o=_ok(cv, r.get("gold_answer") or r.get("answer"))
        if st in ("ABSTAIN","UNVERIFIED_FORMULA") or cv is None: AB+=1
        elif st=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif st=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)  # non-numeric gold + numeric candidate rejected = true reject
        else: AB+=1
    return dict(n=n,TA=TA,TC=TC,FR=FR,FA=FA,AB=AB)

PAIR = [("Qwen2.5-7B","mc_calc_qwen2.5-7b","mc_calc_detsmt_v2_qwen2.5-7b"),
        ("Llama-3.1-8B","mc_calc_llama3.1-8b","mc_calc_detsmt_v2_llama3.1-8b")]
rows=[]
for name, llm, det in PAIR:
    for label, d in [("LLM-SMT (autoformalized)", llm), ("det-SMT (ours)", det)]:
        c=_counts(ROOT/"results"/d/"results.jsonl")
        ver=c["TA"]+c["FA"]; dec=ver+c["TC"]+c["FR"]
        rows.append({"Answer model":name, "Formalization":label,
                     "Prec.":round(100*c["TA"]/ver,1) if ver else float("nan"),
                     "FA":c["FA"],
                     "Coverage":round(100*dec/c["n"],1),
                     "Abst.":round(100*c["AB"]/c["n"],1)})
smt_ablation = pd.DataFrame(rows)
print(smt_ablation.to_string(index=False))
smt_ablation


## Table 4 — Authority and outcome breakdown

Every question assigned to the authority that decided it (or the failure source that caused abstention), with accept/reject/abstain and false accepts per source. FilingCalc is entirely linkbase-authorized by construction; FinanceBench has no calculation linkbase and runs on the registry and filing-specific definitions. FA=0 in every source; the solver never returns UNKNOWN.

In [ ]:
import json, re, collections, statistics
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])
def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out
def _rid(r): return r.get("id") or r.get("financebench_id")
def _ok(rid,cv,g):
    adjudicated = _adjudicated_ok(rid, cv)
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    if cv is None: return None
    t=_nums(g); return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None

ROW_ORDER = ["XBRL calculation linkbase","Operation explicit in question","Metric registry",
             "Filing-specific definition","Missing or ambiguous operand","Unsupported formula",
             "Period or unit ambiguity","Candidate-generation failure","Solver unknown"]

def _nocand_ids(baseline_path):
    """Questions with no parseable candidate answer, measured from the raw-LLM
    baseline (which parses every candidate). This is the candidate-generation
    failure floor shared by all methods, independent of any verifier."""
    return {_rid(json.loads(l)) for l in Path(baseline_path).read_text().splitlines()
            if l.strip() and json.loads(l).get("claimed_value") is None}

def _row_of(r, nocand_ids):
    st=r.get("status"); fs=r.get("formula_source") or ""
    fr=(r.get("failure_reason") or "").split(":")[0]
    if st in ("VERIFIED","VIOLATED"):
        return {"xbrl_linkbase":"XBRL calculation linkbase","policy_registry":"Metric registry",
                "document_derived":"Filing-specific definition",
                "generic_operation":"Operation explicit in question"}.get(fs,"Filing-specific definition")
    # non-decided: a shared no-candidate question is a candidate-generation failure,
    # regardless of what the verifier would otherwise have reported.
    if _rid(r) in nocand_ids: return "Candidate-generation failure"
    if r.get("solver_status")=="UNKNOWN": return "Solver unknown"
    if fr=="": return "Unsupported formula" if fs=="no_formula" else "Candidate-generation failure"
    return {"facts_not_found":"Missing or ambiguous operand",
            "generic_operation_authority_unresolved":"Operation explicit in question",
            "untrusted_formula_source":"Unsupported formula","no_formula":"Unsupported formula",
            "ambiguous_operation":"Unsupported formula","no_numeric_answer":"Candidate-generation failure",
            "period_ambiguity":"Period or unit ambiguity","unit_ambiguity":"Period or unit ambiguity",
            }.get(fr,"Unsupported formula")

def authority_table(path, baseline_path):
    nocand = _nocand_ids(baseline_path)
    recs=[json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]
    agg=collections.defaultdict(lambda: collections.Counter())
    for r in recs:
        row=_row_of(r, nocand); st=r.get("status"); cv=r.get("claimed_value")
        a=agg[row]; a["Questions"]+=1
        if st=="VERIFIED":
            a["Accept"]+=1
            if _ok(_rid(r), cv, r.get("gold_answer") or r.get("answer")) is False: a["FA"]+=1
        elif st=="VIOLATED": a["Reject"]+=1
        else: a["Abstain"]+=1
    rows=[]
    for name in ROW_ORDER:
        a=agg.get(name,collections.Counter())
        rows.append({"Authority / failure source":name,"Questions":a["Questions"],
                     "Accept":a["Accept"],"Reject":a["Reject"],"Abstain":a["Abstain"],"FA":a["FA"]})
    df=pd.DataFrame(rows)
    tot={"Authority / failure source":"Total", **{c:df[c].sum() for c in ["Questions","Accept","Reject","Abstain","FA"]}}
    return pd.concat([df, pd.DataFrame([tot])], ignore_index=True)

authority_filingcalc = authority_table(ROOT/"results/mc_calc_detsmt_v2_haiku/results.jsonl",
                                       ROOT/"results/onepool/xbrl_none/results.jsonl")
authority_financebench = authority_table(ROOT/"results/fb_detsmt_v2_haiku/results.jsonl",
                                         ROOT/"results/onepool/fb_none/results.jsonl")
print("== FilingCalc (n=600) =="); print(authority_filingcalc.to_string(index=False))
print("\n== FinanceBench (n=67) =="); print(authority_financebench.to_string(index=False))

for label, d in [("FilingCalc","mc_calc_detsmt_v2_haiku"),("FinanceBench","fb_detsmt_v2_haiku")]:
    recs=[json.loads(l) for l in (ROOT/"results"/d/"results.jsonl").read_text().splitlines() if l.strip()]
    ss=collections.Counter(r.get("solver_status") or "(none)" for r in recs)
    lat=sorted(float(r["elapsed_s"]) for r in recs if r.get("elapsed_s") is not None)
    print(f"\n{label}: solver UNKNOWN={ss.get('UNKNOWN',0)} | SAT={ss.get('SAT',0)} UNSAT={ss.get('UNSAT',0)}"
          f" | latency median={statistics.median(lat):.1f}s p95={lat[int(0.95*len(lat))-1]:.1f}s")
authority_financebench


### Candidate-generation failures vs. verifier abstentions (FinanceBench)

The abstention floor shared by all methods: questions with no parseable candidate answer. Measured from the raw-LLM baseline, which parses every candidate. VeriFiQA's higher abstention adds its own verifier abstentions (a candidate existed but no authority licensed it).

In [ ]:
import json
from pathlib import Path
def _rid(r): return r.get("id") or r.get("financebench_id")
def load(d): return [json.loads(l) for l in (ROOT/"results"/d/"results.jsonl").read_text().splitlines() if l.strip()]

nocand = {_rid(r) for r in load("onepool/fb_none") if r.get("claimed_value") is None}
print(f"Candidate-generation failures (no parseable candidate): {len(nocand)}/67 = {100*len(nocand)/67:.1f}%")
print("  -> this is the entire abstention of every baseline method")

vq = load("fb_detsmt_v2_haiku")
nd = [r for r in vq if r.get("status") not in ("VERIFIED","VIOLATED")]
cg = [r for r in nd if _rid(r) in nocand]
va = [r for r in nd if _rid(r) not in nocand]
print(f"\nVeriFiQA non-decisions: {len(nd)}/67 = {100*len(nd)/67:.1f}%")
print(f"  candidate-generation failures: {len(cg)}")
print(f"  verifier abstentions (candidate existed, no authorized formula/grounding): {len(va)}")
recovered = nocand - {_rid(r) for r in nd}
if recovered:
    print(f"  note: VeriFiQA decided {len(recovered)} baseline-no-candidate question(s) "
          f"via a parsed value: {sorted(recovered)}")


## Table 5 — Dataset composition (FilingCalc, from file)

The 600 questions by primary statement, counted from the run file rather than hardcoded.

In [ ]:
import json, collections
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
SRC = ROOT / "data/multicompany_provable/data/provable_calcrequired.jsonl"
if not SRC.exists() and (ROOT.parent/SRC.relative_to(ROOT)).exists():
    SRC = ROOT.parent / SRC.relative_to(ROOT)

# FilingCalc's 15 subtotal concepts grouped by primary statement.
STMT = {
 "Income Statement": {"NetIncomeLoss",
   "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
   "OperatingIncomeLoss","GrossProfit","OperatingExpenses"},
 "Balance Sheet": {"AssetsCurrent","StockholdersEquity","LiabilitiesCurrent",
   "PropertyPlantAndEquipmentNet","Liabilities","LiabilitiesNoncurrent","AssetsNoncurrent"},
 "Cash Flow": {"NetCashProvidedByUsedInOperatingActivities",
   "NetCashProvidedByUsedInInvestingActivities","NetCashProvidedByUsedInFinancingActivities"},
}
c2s = {c:s for s,cs in STMT.items() for c in cs}

per = collections.Counter(); metrics_in = collections.defaultdict(set)
for line in SRC.read_text().splitlines():
    if not line.strip(): continue
    concept = json.loads(line)["ground_truth"]["concept"]
    s = c2s.get(concept)
    if s: per[s]+=1; metrics_in[s].add(concept)

rows=[]
for s in ["Income Statement","Balance Sheet","Cash Flow"]:
    rows.append({"Statement": f"{s} ({len(metrics_in[s])})", "Questions": per[s]})
rows.append({"Statement": f"Total ({sum(len(v) for v in metrics_in.values())} metrics)",
             "Questions": sum(per.values())})
dataset_composition = pd.DataFrame(rows)
print(dataset_composition.to_string(index=False))
dataset_composition


## Appendix A — Per-metric coverage (FilingCalc)

FA=0 holds for every one of the 15 subtotal metrics, not only in aggregate.

In [ ]:
import json, re, collections
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out
def _ok(cv,g):
    if cv is None: return None
    t=_nums(g)
    return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None

recs=[json.loads(l) for l in (ROOT/"results/mc_calc_detsmt_v2_haiku/results.jsonl").read_text().splitlines() if l.strip()]
agg=collections.defaultdict(lambda: collections.Counter())
for r in recs:
    base=re.sub(r"_fy\d{4}.*$","",r.get("metric","?"))
    st=r.get("status"); cv=r.get("claimed_value")
    o=_ok(cv, r.get("gold_answer") or r.get("answer"))
    a=agg[base]; a["n"]+=1
    if st in ("ABSTAIN","UNVERIFIED_FORMULA") or cv is None: a["abstain"]+=1
    elif st=="VERIFIED": a["FA" if not o else "verified_ok"]+=1
    elif st=="VIOLATED": a["violated"]+=1
rows=[]
for m,a in sorted(agg.items(), key=lambda kv:-kv[1]["n"]):
    dec=a["n"]-a["abstain"]
    rows.append({"metric":m.replace("_"," "), "n":a["n"],
                 "verified":a["verified_ok"], "violated":a["violated"],
                 "abstain":a["abstain"], "FA":a["FA"],
                 "coverage %":round(100*dec/a["n"],1)})
per_metric = pd.DataFrame(rows)
print(per_metric.to_string(index=False))
print("\nTotal FA across all metrics:", per_metric["FA"].sum())
per_metric


## Appendix B — Metric registry coverage

The curated registry of authorized formulas by source: what lets VerifiQA authorize derived metrics beyond linkbase subtotals.

In [ ]:
import json, collections
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT/"src").exists() and (ROOT.parent/"src").exists():
    ROOT = ROOT.parent
reg=json.load(open(ROOT/"src/verifiqa/policy/data/metric_policies.json"))
by=collections.defaultdict(list)
for e in reg:
    by[e.get("source_type","?")].append(e["metric"])
rows=[]
for src,ms in sorted(by.items(), key=lambda kv:-len(kv[1])):
    ex=", ".join(sorted(m.replace("_"," ") for m in ms)[:4])
    rows.append({"source":src, "policies":len(ms), "example metrics":ex+(" ..." if len(ms)>4 else "")})
registry_coverage = pd.DataFrame(rows)
print("Total policies:", len(reg))
print(registry_coverage.to_string(index=False))
registry_coverage


In [ ]:
# ICAIF main-paper figure: per-statement-family error profile on XBRLBench.
#
# Bars are rates *within* a statement family (denominators differ across
# families); family denominators are carried in the legend so raw counts stay
# recoverable. VeriFin's detection outcome is constant (all incorrect answers
# flagged, 0 false accepts, 0 unresolved) and belongs in the caption, not in
# bar heights a reader would misread as verifier recall.
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.font_manager import findfont, FontProperties
from matplotlib.patches import Patch

# --------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------

# Keep each display label attached to its result path so adding a model cannot
# silently misalign bars, tick labels, and source files.
MODEL_SPECS = [
    (
        "GPT-5.5",
        "GPT-5.5",
        "results/latest/mc_calc_gpt5.5_codex_det_smt_Nrepair_run1/results.jsonl",
    ),
    (
        "Claude Haiku 4.5",
        "Claude\nHaiku 4.5",
        "results/mc_calc_detsmt_v2_haiku/results.jsonl",
    ),
    (
        "Fino1",
        "Fin-o1",
        "results/latest/mc-fino-30b-latest/results.jsonl",
    ),
    (
        "Qwen3-30B",
        "Qwen3\n30B",
        "results/latest/mc-qwen-30b-latest/results.jsonl",
    ),
    (
        "Llama-3.1-8B",
        "Llama3.1\n8B",
        "results/mc_calc_detsmt_v2_llama3.1-8b/results.jsonl",
    ),
    (
        "Qwen2.5-7B",
        "Qwen2.5\n7B",
        "results/mc_calc_detsmt_v2_qwen2.5-7b/results.jsonl",
    ),
]
MODEL_ORDER = [model for model, _, _ in MODEL_SPECS]
MODEL_TICK_LABELS = [tick for _, tick, _ in MODEL_SPECS]


def paper_model_runs(root):
    return [(model, root / relative_path) for model, _, relative_path in MODEL_SPECS]


def _read_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text().splitlines()
        if line.strip()
    ]


def resolve_root():
    """Find the repository from local, Colab, or mounted-Drive locations."""
    candidates = []
    configured_root = globals().get("ROOT")
    if configured_root is not None:
        candidates.append(Path(configured_root))

    cwd = Path.cwd()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "verifiqa",
            Path("/content/verifiqa"),
            Path("/content/drive/MyDrive/verifiqa"),
        ]
    )

    # Also detect a repository placed one directory below common Colab roots.
    for base in [Path("/content"), Path("/content/drive/MyDrive")]:
        if base.exists():
            for marker in base.glob(
                "*/results/latest/"
                "mc_calc_gpt5.5_codex_det_smt_Nrepair_run1/results.jsonl"
            ):
                candidates.append(marker.parents[3])

    checked = []
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(candidate)
        if all(path.is_file() for _, path in paper_model_runs(candidate)):
            return candidate

    checked_text = "\n".join(f"  - {path}" for path in checked)
    raise FileNotFoundError(
        "Could not locate the six XBRLBench result files. Set ROOT to the "
        "verifiqa repository directory before running this cell. Checked:\n"
        f"{checked_text}"
    )


def load_questions(root):
    """Load the canonical question file, or reuse question text in a result run."""
    dataset_path = (
        root / "data/multicompany_provable/data/provable_calcrequired.jsonl"
    )
    if dataset_path.is_file():
        return {
            row["financebench_id"]: row
            for row in _read_jsonl(dataset_path)
        }

    # Every result row carries the same question text and ID, so the figure does
    # not need a second copy of the dataset merely to assign statement families.
    reference_path = paper_model_runs(root)[0][1]
    reference_rows = _read_jsonl(reference_path)
    questions = {
        row["id"]: {"financebench_id": row["id"], "question": row["question"]}
        for row in reference_rows
        if row.get("id") and row.get("question")
    }
    if len(questions) != 600:
        raise ValueError(
            f"Expected 600 embedded questions in {reference_path}; "
            f"found {len(questions)}"
        )
    warnings.warn(
        f"{dataset_path} was not found; using question text embedded in "
        f"{reference_path}.",
        stacklevel=2,
    )
    return questions


FAMILY_TERMS = [
    ("gross profit", "Income statement"),
    ("operating income", "Income statement"),
    ("operating expenses", "Income statement"),
    ("income before income taxes", "Income statement"),
    ("net income", "Income statement"),
    ("total current assets", "Balance sheet"),
    ("total non-current assets", "Balance sheet"),
    ("total current liabilities", "Balance sheet"),
    ("total non-current liabilities", "Balance sheet"),
    ("total liabilities", "Balance sheet"),
    ("total shareholders", "Balance sheet"),
    ("net property", "Balance sheet"),
    ("net cash from operating", "Cash flow"),
    ("net cash from investing", "Cash flow"),
    ("net cash from financing", "Cash flow"),
]
FAMILY_ORDER = ["Income statement", "Balance sheet", "Cash flow"]


def _statement_family(question):
    lowered = question.lower()
    for term, family in FAMILY_TERMS:
        if term in lowered:
            return family
    raise ValueError(f"Unmapped XBRLBench question: {question}")


def _paper_numbers(text):
    values = []
    for token in re.findall(r"-?\d[\d,]*\.?\d*", str(text or "")):
        try:
            values.append(abs(float(token.replace(",", ""))))
        except ValueError:
            pass
    return values


def _paper_correct(claimed_value, gold):
    """Paper convention: magnitude match within max($1m, 1%)."""
    if claimed_value is None:
        return False
    gold_values = _paper_numbers(gold)
    return any(
        abs(abs(float(claimed_value)) - value) <= max(1.0, value * 0.01)
        for value in gold_values
    )


def family_denominators(questions):
    """Questions per statement family. Constant across models by construction."""
    totals = {family: 0 for family in FAMILY_ORDER}
    for row in questions.values():
        totals[_statement_family(row["question"])] += 1
    return totals


def _xbrl_caught_by_family(path, questions):
    caught = {family: 0 for family in FAMILY_ORDER}
    incorrect_numeric = {family: 0 for family in FAMILY_ORDER}
    false_accepts = 0
    unresolved_incorrect = 0
    rows = [
        json.loads(line)
        for line in Path(path).read_text().splitlines()
        if line.strip()
    ]
    assert len(rows) == 600
    assert len({row["id"] for row in rows}) == 600
    assert all(row["id"] in questions for row in rows)
    for row in rows:
        claimed_value = row.get("claimed_value")
        if claimed_value is None or _paper_correct(
            claimed_value,
            row.get("gold_answer") or row.get("answer"),
        ):
            continue
        family = _statement_family(questions[row["id"]]["question"])
        incorrect_numeric[family] += 1
        if row.get("status") == "VIOLATED":
            caught[family] += 1
        elif row.get("status") == "VERIFIED":
            false_accepts += 1
        else:
            unresolved_incorrect += 1
    return caught, incorrect_numeric, false_accepts, unresolved_incorrect


# --------------------------------------------------------------------------
# Figure
# --------------------------------------------------------------------------

XBRL_SPECS = [
    ("Income statement", "#28618F", "#1D496C", ""),
    ("Balance sheet", "#2A8C82", "#1C625B", "///"),
    ("Cash flow", "#D89B2B", "#8C631B", "..."),
]
INK, MUTED, GRID = "#17212B", "#5B6573", "#D4DAE1"

# ACM sigconf: draw at the final full-text width. Large sans-serif type and
# a taller plotting area remain readable after the figure is placed in paper.
FIG_WIDTH_IN = 8.50
FIG_HEIGHT_IN = 4.55

# Sans-serif chart labels reproduce more clearly than small serif labels.
SANS_STACK = ["Arial", "Helvetica Neue", "Helvetica", "DejaVu Sans"]


def _warn_on_font_fallback():
    resolved = findfont(FontProperties(family="Arial"), fallback_to_default=True)
    if "DejaVu" in resolved:
        warnings.warn(
            "Arial is unavailable; matplotlib fell back to DejaVu Sans "
            f"({resolved}).",
            stacklevel=2,
        )


def make_figure(caught, totals, output_stem):
    """caught: DataFrame indexed by model, columns = FAMILY_ORDER (counts).
    totals: dict family -> number of questions in that family (per model).
    """
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    _warn_on_font_fallback()
    with plt.rc_context(
        {
            "font.family": "sans-serif",
            "font.sans-serif": SANS_STACK,
            "font.size": 11.5,
            "axes.linewidth": 0.90,
            "hatch.linewidth": 0.90,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "figure.dpi": 300,
            "savefig.dpi": 1200,
        }
    ):
        fig, ax = plt.subplots(
            figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN), layout="constrained"
        )
        ordered = caught.reindex(MODEL_ORDER)
        # Wide bars retain the cleaner original layout, while a modest gap
        # between model groups keeps adjacent models visually distinct.
        model_x = np.arange(len(MODEL_ORDER)) * 1.60
        bar_width = 0.36
        offsets = (-0.40, 0.0, 0.40)

        rate_table = pd.DataFrame(
            {
                column: 100.0 * ordered[column] / totals[column]
                for column, *_ in XBRL_SPECS
            },
            index=ordered.index,
        )

        # Keep every value label visually attached to its own bar.
        label_y = rate_table + 1.0

        for (column, facecolor, edgecolor, hatch), offset in zip(
            XBRL_SPECS, offsets
        ):
            counts = ordered[column].to_numpy()
            rates = rate_table[column].to_numpy()
            bars = ax.bar(
                model_x + offset,
                rates,
                width=bar_width,
                color=facecolor,
                edgecolor=edgecolor,
                linewidth=0.95,
                hatch=hatch,
                zorder=3,
            )
            for model, bar, rate, count in zip(
                MODEL_ORDER, bars, rates, counts
            ):
                label_height = float(label_y.loc[model, column])
                if np.isclose(rate, 0.0):
                    rate_text = "0%"
                else:
                    rate_text = f"{rate:.1f}%" if rate < 10.0 else f"{rate:.0f}%"
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    label_height,
                    f"{rate_text}\n({int(count)})",
                    ha="center",
                    va="bottom",
                    fontsize=9.0 if model == "GPT-5.5" else 8.2,
                    color=INK,
                    fontweight="normal" if model == "GPT-5.5" else "normal",
                    linespacing=0.90,
                    zorder=4,
                )

        ax.set_xlim(model_x[0] - 0.86, model_x[-1] + 0.86)
        max_label_y = float(label_y.to_numpy().max())
        ax.set_ylim(0, np.ceil((max_label_y + 5.0) / 5.0) * 5.0)
        ax.set_ylabel(
            "Wrong Claims Caught by VeriFin (%)",
            fontsize=13.0,
            fontweight="normal",
            color=INK,
            labelpad=9,
        )
        ax.set_xticks(model_x, MODEL_TICK_LABELS, fontsize=10.5,
                      fontweight="normal", color=INK)
        ax.grid(axis="y", color=GRID, linewidth=0.75, alpha=0.9, zorder=1)
        ax.tick_params(axis="x", length=0, pad=8)
        ax.tick_params(axis="y", length=3.2, width=0.80, colors=MUTED,
                       labelsize=11.0)
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.spines["bottom"].set_color("#AEB5BE")

        # Denominators live in the legend, so raw counts stay recoverable.
        legend_handles = [
            Patch(
                facecolor=facecolor,
                edgecolor=edgecolor,
                linewidth=0.95,
                hatch=hatch,
                label=f"{label} ($n={totals[label]}$)",
            )
            for label, facecolor, edgecolor, hatch in XBRL_SPECS
        ]
        ax.legend(
            handles=legend_handles,
            loc="lower center",
            bbox_to_anchor=(0.5, 1.01),
            ncol=3,
            frameon=False,
            fontsize=10.5,
            handlelength=1.55,
            handletextpad=0.55,
            columnspacing=1.25,
            borderaxespad=0,
        )

        export_options = {
            "bbox_inches": "tight",
            "pad_inches": 0.02,
            "facecolor": "white",
        }
        fig.savefig(f"{output_stem}.pdf", **export_options)
        fig.savefig(f"{output_stem}.svg", **export_options)
        fig.savefig(f"{output_stem}.png", dpi=1200, **export_options)
        return fig


def main(root):
    questions = load_questions(root)
    totals = family_denominators(questions)
    caught_rows, incorrect_rows = [], []
    false_accepts, unresolved = {}, {}
    for model, path in paper_model_runs(root):
        c, i, fa, un = _xbrl_caught_by_family(path, questions)
        caught_rows.append({"Answer model": model, **c})
        incorrect_rows.append({"Answer model": model, **i})
        false_accepts[model], unresolved[model] = fa, un

    xbrl_caught = pd.DataFrame(caught_rows).set_index("Answer model")
    xbrl_incorrect = pd.DataFrame(incorrect_rows).set_index("Answer model")
    assert xbrl_caught[FAMILY_ORDER].equals(xbrl_incorrect[FAMILY_ORDER])
    assert all(v == 0 for v in false_accepts.values())
    assert all(v == 0 for v in unresolved.values())
    assert xbrl_caught.index.tolist() == MODEL_ORDER
    assert sum(totals.values()) == 600

    fig = make_figure(
        xbrl_caught[FAMILY_ORDER],
        totals,
        str(root / "figures/xbrl_error_profile_by_statement_family"),
    )
    plt.show()
    return fig


ROOT = resolve_root()
_figure = main(ROOT)

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
nan=float("nan")

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

ADJUDICATED_TARGETS = {
    "financebench_id_04672": (8700.0,),
    "financebench_id_04980": (4600.0,),
    "financebench_id_00882": (8400.0,),
    "financebench_id_02024": (1959.0,),
    "financebench_id_01930": (0.0,),
    "financebench_id_01936": (87.0, 93.0),
}
_NO_ADJUDICATION = object()
def _adjudicated_ok(rid, cv):
    if rid not in ADJUDICATED_TARGETS:
        return _NO_ADJUDICATION
    if cv is None:
        return None
    return any(abs(abs(float(cv)) - abs(target)) <= max(1.0, 0.01 * abs(target))
               for target in ADJUDICATED_TARGETS[rid])

def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out

def _rid(r): return r.get("id") or r.get("financebench_id")

def _ok(rid, cv, g, kc):
    adjudicated = _adjudicated_ok(rid, cv) if kc else _NO_ADJUDICATION
    if adjudicated is not _NO_ADJUDICATION: return adjudicated
    if cv is None: return None
    t=_nums(g)
    return any(abs(abs(cv)-x)<=max(1.0,x*0.01) for x in t) if t else None

def _parseable_ids(path):
    """Answerable questions: those with a parseable numeric candidate, measured from the
    raw-LLM baseline (which parses every candidate). Questions with no numeric candidate
    are candidate-generation failures, shared by all methods and excluded from abstention."""
    ids=set()
    for line in Path(path).read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line)
        if r.get("claimed_value") is not None: ids.add(_rid(r))
    return ids

def _score(path, kc, parseable):
    TA=TC=FR=FA=0; status={}
    for line in Path(path).read_text().splitlines():
        if not line.strip(): continue
        r=json.loads(line); rid=_rid(r); st=r.get("status"); cv=r.get("claimed_value")
        status[rid]=st
        if cv is None: continue                      # no candidate: not a decision
        o=_ok(rid, cv, r.get("gold_answer") or r.get("answer"), kc)
        if st=="VERIFIED": TA,FA=(TA+1,FA) if o else (TA,FA+1)
        elif st=="VIOLATED": TC,FR=(TC+1,FR) if (o is False or o is None) else (TC,FR+1)
    ver=TA+FA; dec=ver+TC+FR
    # Abstention = AB/n over all benchmark questions, matching the paper's metric
    # definition. Questions with no parseable candidate count as declined.
    nd_all=sum(1 for st in status.values() if st not in ("VERIFIED","VIOLATED"))
    return {"Precision": 100*TA/ver if ver else nan,
            "Accuracy": 100*(TA+TC)/dec if dec else nan,
            "Abstention": 100*nd_all/len(status) if status else nan,
            "FA": FA}

DATASETS = [
    ("XBRLBench", False, {
        "raw-LLM":"onepool/xbrl_none", "LLM-judge":"onepool/xbrl_judge",
        "judge+formula":"onepool/xbrl_judge_formula", "PoT":"onepool/xbrl_pot",
        "VeriFin (ours)":"mc_calc_detsmt_v2_haiku"}),
    ("FinanceBench", True, {
        "raw-LLM":"onepool/fb_none", "LLM-judge":"onepool/fb_judge",
        "judge+formula":"onepool/fb_judge_formula", "PoT":"onepool/fb_pot",
        "VeriFin (ours)":"fb_detsmt_v2_haiku"}),
]
METHODS = ["raw-LLM","LLM-judge","judge+formula","PoT","VeriFin (ours)"]

cols={}
answerable={}
for dset, kc, runs in DATASETS:
    parseable=_parseable_ids(ROOT/"results"/runs["raw-LLM"]/"results.jsonl")
    answerable[dset]=len(parseable)
    for m in METHODS:
        s=_score(ROOT/"results"/runs[m]/"results.jsonl", kc, parseable)
        for metric,val in s.items():
            cols.setdefault((dset,metric),{})[m]=val

main_results = pd.DataFrame(cols).reindex(METHODS)
main_results.columns = pd.MultiIndex.from_tuples(main_results.columns)
print("answerable (parseable-candidate) questions:", answerable)
def _fmt(v,c): return f"{int(v)}" if c[1]=="FA" else f"{v:.1f}"
print(pd.DataFrame({c:[_fmt(v,c) for v in main_results[c]] for c in main_results.columns},
                   index=METHODS).to_string())

# ---- paper LaTeX (tab:main), emitted from the same numbers ----
def _cell(m, dset, metric, bold_if_best=False):
    v=main_results.loc[m,(dset,metric)]
    s=f"{int(v)}" if metric=="FA" else f"{v:.1f}"
    return s
best={("XBRLBench","Precision"),("XBRLBench","Accuracy"),("XBRLBench","FA"),
      ("FinanceBench","Precision"),("FinanceBench","FA")}   # cells VeriFin wins
lines=[]
for m in METHODS:
    cells_=[]
    for dset in ["XBRLBench","FinanceBench"]:
        for metric in ["Precision","Accuracy","Abstention","FA"]:
            s=_cell(m,dset,metric)
            if m=="VeriFin (ours)" and (dset,metric) in best: s=f"\\bfseries {s}"
            cells_.append(s)
    row="  & "+" & ".join(cells_)+" \\\\"
    if m=="VeriFin (ours)": row="\\rowcolor{ourshl}\n"+m+row
    else: row=m+row
    lines.append(row)
print("\n% ---- tab:main LaTeX (auditable, from result files) ----")
print("\n".join(lines))
main_results


In [ ]:
"""ICAIF main-paper figure: per-statement-family error profile on XBRLBench.

Bars are error rates *within* a statement family, so denominators differ across
families; family denominators are carried in the legend to keep raw counts
recoverable. VeriFin's detection outcome is constant across the run (every
incorrect answer flagged, zero false accepts, zero unresolved) and therefore
belongs in the caption rather than in bar heights, which a reader would
otherwise misread as verifier recall.
"""

import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.patches import Patch

# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------

# Display label, tick label, and result path stay bound together so adding a
# model cannot silently misalign bars, ticks, and source files.
MODEL_SPECS = [
    ("GPT-5.5", "GPT-5.5",
     "results/latest/mc_calc_gpt5.5_codex_det_smt_Nrepair_run1/results.jsonl"),
    ("Claude Haiku 4.5", "Claude\nHaiku 4.5",
     "results/mc_calc_detsmt_v2_haiku/results.jsonl"),
    ("Fino1", "Fin-o1",
     "results/latest/mc-fino-30b-latest/results.jsonl"),
    ("Qwen3-30B", "Qwen3\n30B",
     "results/latest/mc-qwen-30b-latest/results.jsonl"),
    ("Llama-3.1-8B", "Llama3.1\n8B",
     "results/mc_calc_detsmt_v2_llama3.1-8b/results.jsonl"),
    ("Qwen2.5-7B", "Qwen2.5\n7B",
     "results/mc_calc_detsmt_v2_qwen2.5-7b/results.jsonl"),
]
MODEL_ORDER = [model for model, _, _ in MODEL_SPECS]
MODEL_TICK_LABELS = [tick for _, tick, _ in MODEL_SPECS]

DATASET_RELPATH = "data/multicompany_provable/data/provable_calcrequired.jsonl"
N_QUESTIONS = 600

FAMILY_TERMS = [
    ("gross profit", "Income statement"),
    ("operating income", "Income statement"),
    ("operating expenses", "Income statement"),
    ("income before income taxes", "Income statement"),
    ("net income", "Income statement"),
    ("total current assets", "Balance sheet"),
    ("total non-current assets", "Balance sheet"),
    ("total current liabilities", "Balance sheet"),
    ("total non-current liabilities", "Balance sheet"),
    ("total liabilities", "Balance sheet"),
    ("total shareholders", "Balance sheet"),
    ("net property", "Balance sheet"),
    ("net cash from operating", "Cash flow"),
    ("net cash from investing", "Cash flow"),
    ("net cash from financing", "Cash flow"),
]
FAMILY_ORDER = ["Income statement", "Balance sheet", "Cash flow"]

# label, face, edge, hatch
FAMILY_STYLES = [
    ("Income statement", "#28618F", "#1D496C", ""),
    ("Balance sheet", "#2A8C82", "#1C625B", "///"),
    ("Cash flow", "#D89B2B", "#8C631B", "..."),
]
INK, MUTED, GRID = "#17212B", "#5B6573", "#D4DAE1"

# ACM sigconf: draw at final full-text width so type stays readable in place.
FIG_WIDTH_IN, FIG_HEIGHT_IN = 8.50, 4.55
SANS_STACK = ["Arial", "Helvetica Neue", "Helvetica", "DejaVu Sans"]

BAR_WIDTH = 0.36
GROUP_STEP = 1.60
BAR_OFFSETS = (-0.40, 0.0, 0.40)
LABEL_FONTSIZE = 8.2
LABEL_LINESPACING = 0.90
LABEL_CLEARANCE_PT = 2.5   # small gap above each bar
LABEL_GAP_SLACK = 1.12     # multiplier on text height for label-to-label gap


# --------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------

def read_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text().splitlines()
        if line.strip()
    ]


def model_runs(root):
    return [(model, root / relpath) for model, _, relpath in MODEL_SPECS]


def resolve_root():
    """Find the repository from local, Colab, or mounted-Drive locations."""
    candidates = []
    configured = globals().get("ROOT")
    if configured is not None:
        candidates.append(Path(configured))

    cwd = Path.cwd()
    candidates += [
        cwd, *cwd.parents,
        cwd / "verifiqa",
        Path("/content/verifiqa"),
        Path("/content/drive/MyDrive/verifiqa"),
    ]

    # Also detect a repository placed one level below common Colab roots.
    probe = MODEL_SPECS[0][2]
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            candidates += [m.parents[3] for m in base.glob(f"*/{probe}")]

    checked, seen = [], set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(candidate)
        if all(path.is_file() for _, path in model_runs(candidate)):
            return candidate

    listing = "\n".join(f"  - {path}" for path in checked)
    raise FileNotFoundError(
        f"Could not locate the {len(MODEL_SPECS)} XBRLBench result files. Set "
        f"ROOT to the verifiqa repository directory. Checked:\n{listing}"
    )


def load_questions(root):
    """Load the canonical question file, or reuse question text from a run."""
    dataset_path = root / DATASET_RELPATH
    if dataset_path.is_file():
        return {row["financebench_id"]: row for row in read_jsonl(dataset_path)}

    # Result rows carry the same question text and ID, so the figure does not
    # need a second copy of the dataset merely to assign statement families.
    reference = model_runs(root)[0][1]
    questions = {
        row["id"]: {"financebench_id": row["id"], "question": row["question"]}
        for row in read_jsonl(reference)
        if row.get("id") and row.get("question")
    }
    if len(questions) != N_QUESTIONS:
        raise ValueError(
            f"Expected {N_QUESTIONS} embedded questions in {reference}; "
            f"found {len(questions)}"
        )
    warnings.warn(
        f"{dataset_path} not found; using question text embedded in "
        f"{reference}.",
        stacklevel=2,
    )
    return questions


def statement_family(question):
    lowered = question.lower()
    for term, family in FAMILY_TERMS:
        if term in lowered:
            return family
    raise ValueError(f"Unmapped XBRLBench question: {question}")


def parse_numbers(text):
    values = []
    for token in re.findall(r"-?\d[\d,]*\.?\d*", str(text or "")):
        try:
            values.append(abs(float(token.replace(",", ""))))
        except ValueError:
            pass
    return values


def is_correct(claimed_value, gold):
    """Paper convention: magnitude match within max($1m, 1%)."""
    if claimed_value is None:
        return False
    return any(
        abs(abs(float(claimed_value)) - value) <= max(1.0, value * 0.01)
        for value in parse_numbers(gold)
    )


def family_denominators(questions):
    """Questions per statement family. Constant across models by construction."""
    totals = dict.fromkeys(FAMILY_ORDER, 0)
    for row in questions.values():
        totals[statement_family(row["question"])] += 1
    return totals


def tally_run(path, questions):
    """Return (caught, incorrect, false_accepts, unresolved) for one model."""
    caught = dict.fromkeys(FAMILY_ORDER, 0)
    incorrect = dict.fromkeys(FAMILY_ORDER, 0)
    false_accepts = unresolved = 0

    rows = read_jsonl(path)
    assert len(rows) == N_QUESTIONS, f"{path}: {len(rows)} rows"
    assert len({row["id"] for row in rows}) == N_QUESTIONS
    assert all(row["id"] in questions for row in rows)

    for row in rows:
        claimed = row.get("claimed_value")
        gold = row.get("gold_answer") or row.get("answer")
        if claimed is None or is_correct(claimed, gold):
            continue
        family = statement_family(questions[row["id"]]["question"])
        incorrect[family] += 1
        status = row.get("status")
        if status == "VIOLATED":
            caught[family] += 1
        elif status == "VERIFIED":
            false_accepts += 1
        else:
            unresolved += 1

    return caught, incorrect, false_accepts, unresolved


# --------------------------------------------------------------------------
# Figure
# --------------------------------------------------------------------------

def warn_on_font_fallback():
    resolved = findfont(FontProperties(family="Arial"), fallback_to_default=True)
    if "DejaVu" in resolved:
        warnings.warn(
            f"Arial unavailable; matplotlib fell back to DejaVu Sans "
            f"({resolved}).",
            stacklevel=2,
        )


def near_bar_label_heights(rates, clearance):
    """Place every numeric label a small, fixed distance above its bar."""
    return rates.astype(float) + clearance


def _label_geometry(ax, y_top):
    """Convert label text height and clearance from points into data units."""
    axes_height_pt = ax.get_window_extent().height * 72.0 / ax.figure.dpi
    pt_per_unit = axes_height_pt / y_top
    text_height_pt = LABEL_FONTSIZE * (1.0 + LABEL_LINESPACING) + 2.0
    return (
        LABEL_CLEARANCE_PT / pt_per_unit,
        text_height_pt * LABEL_GAP_SLACK / pt_per_unit,
    )


def solve_label_heights(ax, rates, passes=4):
    """Co-solve label heights and the y-limit.

    Spacing depends on the axis range and the range depends on where the
    topmost label lands, so iterate to a fixed point instead of guessing.
    """
    y_top = max(float(rates.to_numpy().max()) * 1.45, 5.0)
    heights = None
    for _ in range(passes):
        ax.set_ylim(0, y_top)
        ax.figure.canvas.draw()
        clearance, gap = _label_geometry(ax, y_top)
        heights = near_bar_label_heights(rates, clearance)
        new_top = np.ceil((float(heights.to_numpy().max()) + gap) / 5.0) * 5.0
        if abs(new_top - y_top) < 1e-6:
            break
        y_top = new_top
    ax.set_ylim(0, y_top)
    return heights


def make_figure(caught, totals, output_stem):
    """caught: DataFrame indexed by model, columns FAMILY_ORDER (counts).
    totals: dict family -> number of questions in that family.
    """
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    warn_on_font_fallback()

    with plt.rc_context({
        "font.family": "sans-serif",
        "font.sans-serif": SANS_STACK,
        "font.size": 11.5,
        "axes.linewidth": 0.90,
        "hatch.linewidth": 0.90,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.dpi": 300,
        "savefig.dpi": 1200,
    }):
        fig, ax = plt.subplots(
            figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN), layout="constrained"
        )
        ordered = caught.reindex(MODEL_ORDER)
        model_x = np.arange(len(MODEL_ORDER)) * GROUP_STEP

        rates = pd.DataFrame(
            {f: 100.0 * ordered[f] / totals[f] for f in FAMILY_ORDER},
            index=ordered.index,
        )
        ax.set_xlim(model_x[0] - 0.86, model_x[-1] + 0.86)

        bar_groups = {}
        for (family, face, edge, hatch), offset in zip(
            FAMILY_STYLES, BAR_OFFSETS
        ):
            bar_groups[family] = ax.bar(
                model_x + offset,
                rates[family].to_numpy(),
                width=BAR_WIDTH,
                color=face,
                edgecolor=edge,
                linewidth=0.95,
                hatch=hatch,
                zorder=3,
            )

        ax.set_ylabel(
            "Wrong claims caught by VeriFin (%)",
            fontsize=13.0,
            fontweight="normal",
            color=INK,
            labelpad=9,
        )
        ax.set_xticks(model_x, MODEL_TICK_LABELS, fontsize=10.5,
                      fontweight="normal", color=INK)
        ax.grid(axis="y", color=GRID, linewidth=0.75, alpha=0.9, zorder=1)
        ax.tick_params(axis="x", length=0, pad=8)
        ax.tick_params(axis="y", length=3.2, width=0.80, colors=MUTED,
                       labelsize=11.0)
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.spines["bottom"].set_color("#AEB5BE")

        # Denominators live in the legend so raw counts stay recoverable.
        ax.legend(
            handles=[
                Patch(facecolor=face, edgecolor=edge, linewidth=0.95,
                      hatch=hatch, label=f"{family} ($n={totals[family]}$)")
                for family, face, edge, hatch in FAMILY_STYLES
            ],
            loc="lower center",
            bbox_to_anchor=(0.5, 1.01),
            ncol=3,
            frameon=False,
            fontsize=10.5,
            handlelength=1.55,
            handletextpad=0.55,
            columnspacing=1.25,
            borderaxespad=0,
        )

        # Legend and ticks are in place, so constrained layout has settled and
        # the axes box will not shrink under the labels after they are solved.
        label_y = solve_label_heights(ax, rates)

        for (family, _, edge, _) in FAMILY_STYLES:
            for model, bar in zip(MODEL_ORDER, bar_groups[family]):
                rate = float(rates.loc[model, family])
                count = int(ordered.loc[model, family])
                centre = bar.get_x() + bar.get_width() / 2
                if np.isclose(rate, 0.0):
                    rate_text = "0%"
                else:
                    rate_text = f"{rate:.1f}%" if rate < 10.0 else f"{rate:.0f}%"
                ax.text(
                    centre,
                    float(label_y.loc[model, family]),
                    f"{rate_text}\n({count})",
                    ha="center",
                    va="bottom",
                    fontsize=9.0 if model == "GPT-5.5" else LABEL_FONTSIZE,
                    color=INK,
                    fontweight="normal" if model == "GPT-5.5" else "normal",
                    linespacing=LABEL_LINESPACING,
                    zorder=5,
                )

        export = {"bbox_inches": "tight", "pad_inches": 0.02,
                  "facecolor": "white"}
        fig.savefig(f"{output_stem}.pdf", **export)
        fig.savefig(f"{output_stem}.svg", **export)
        fig.savefig(f"{output_stem}.png", dpi=1200, **export)
        return fig


# --------------------------------------------------------------------------
# Entry point
# --------------------------------------------------------------------------

def main(root):
    questions = load_questions(root)
    totals = family_denominators(questions)

    caught_rows, incorrect_rows = [], []
    false_accepts, unresolved = {}, {}
    for model, path in model_runs(root):
        caught, incorrect, fa, un = tally_run(path, questions)
        caught_rows.append({"Answer model": model, **caught})
        incorrect_rows.append({"Answer model": model, **incorrect})
        false_accepts[model], unresolved[model] = fa, un

    caught_df = pd.DataFrame(caught_rows).set_index("Answer model")
    incorrect_df = pd.DataFrame(incorrect_rows).set_index("Answer model")

    assert caught_df[FAMILY_ORDER].equals(incorrect_df[FAMILY_ORDER]), \
        "some incorrect answers were not flagged"
    assert all(v == 0 for v in false_accepts.values()), false_accepts
    assert all(v == 0 for v in unresolved.values()), unresolved
    assert caught_df.index.tolist() == MODEL_ORDER
    assert sum(totals.values()) == N_QUESTIONS

    print(f"total incorrect answers flagged: "
          f"{int(caught_df[FAMILY_ORDER].to_numpy().sum())}")

    fig = make_figure(
        caught_df[FAMILY_ORDER],
        totals,
        root / "figures/xbrl_error_profile_by_statement_family",
    )
    plt.show()
    return fig


if __name__ == "__main__":
    ROOT = resolve_root()
    _figure = main(ROOT)

In [ ]:
from __future__ import annotations

import json
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class ModelSpec:
    """A model's table label, x-axis label, and result file."""

    name: str
    tick_label: str
    result_path: str


# Llama 3.1 intentionally precedes Qwen2.5, as requested.
MODEL_SPECS = (
    ModelSpec(
        "GPT-5.5",
        "GPT-5.5",
        "results/latest/mc_calc_gpt5.5_codex_det_smt_Nrepair_run1/results.jsonl",
    ),
    ModelSpec(
        "Claude Haiku 4.5",
        "Claude\nHaiku 4.5",
        "results/mc_calc_detsmt_v2_haiku/results.jsonl",
    ),
    ModelSpec(
        "Fino1",
        "Fin-o1",
        "results/latest/mc-fino-30b-latest/results.jsonl",
    ),
    ModelSpec(
        "Qwen3-30B",
        "Qwen3\n30B",
        "results/latest/mc-qwen-30b-latest/results.jsonl",
    ),
    ModelSpec(
        "Llama-3.1-8B",
        "Llama 3.1\n8B",
        "results/mc_calc_detsmt_v2_llama3.1-8b/results.jsonl",
    ),
    ModelSpec(
        "Qwen2.5-7B",
        "Qwen2.5\n7B",
        "results/mc_calc_detsmt_v2_qwen2.5-7b/results.jsonl",
    ),
)

MODEL_ORDER = [spec.name for spec in MODEL_SPECS]
MODEL_TICK_LABELS = [spec.tick_label for spec in MODEL_SPECS]

DATASET_RELPATH = "data/multicompany_provable/data/provable_calcrequired.jsonl"
N_QUESTIONS = 600

FAMILY_TERMS = (
    ("gross profit", "Income statement"),
    ("operating income", "Income statement"),
    ("operating expenses", "Income statement"),
    ("income before income taxes", "Income statement"),
    ("net income", "Income statement"),
    ("total current assets", "Balance sheet"),
    ("total non-current assets", "Balance sheet"),
    ("total current liabilities", "Balance sheet"),
    ("total non-current liabilities", "Balance sheet"),
    ("total liabilities", "Balance sheet"),
    ("total shareholders", "Balance sheet"),
    ("net property", "Balance sheet"),
    ("net cash from operating", "Cash flow"),
    ("net cash from investing", "Cash flow"),
    ("net cash from financing", "Cash flow"),
)
FAMILY_ORDER = ["Income statement", "Balance sheet", "Cash flow"]

# label, fill, edge, hatch, retained text color field
# The palette remains recognizable in grayscale because hatches are retained.
FAMILY_STYLES = (
    ("Income statement", "#3B6EA8", "#284D76", "", "#FFFFFF"),
    ("Balance sheet", "#3D9085", "#28665E", "//", "#FFFFFF"),
    ("Cash flow", "#C99538", "#805F22", "..", "#3C2C12"),
)

INK = "#18212B"
MUTED = "#65707D"
GRID = "#E2E6EB"
BASELINE = "#AEB6C0"

# ACM-style full text width: the figure is drawn at its final two-column size.
FIG_WIDTH_IN = 7.08
FIG_HEIGHT_IN = 3.30
FONT_STACK = ["Arial", "Helvetica Neue", "Helvetica", "DejaVu Sans"]

BAR_WIDTH = 0.17
BAR_OFFSETS = (-0.30, 0.0, 0.30)
ANNOTATION_FONTSIZE = 6.65
ANNOTATION_PADDING_PT = 3.0
ANNOTATION_LINESPACING = 0.88
Y_AXIS_MAX = 100.0
Y_TICK_STEP = 20.0

_NUMBER_PATTERN = re.compile(r"-?\d[\d,]*\.?\d*")


# ---------------------------------------------------------------------------
# Data loading and validation
# ---------------------------------------------------------------------------


def read_jsonl(path: Path) -> list[dict]:
    """Read a UTF-8 JSONL file, skipping blank lines."""
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def model_runs(root: Path) -> list[tuple[str, Path]]:
    return [(spec.name, root / spec.result_path) for spec in MODEL_SPECS]


def resolve_root() -> Path:
    """Locate the repository locally, in Colab, or on mounted Google Drive."""
    candidates: list[Path] = []
    configured = globals().get("ROOT")
    if configured is not None:
        candidates.append(Path(configured))

    cwd = Path.cwd()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "verifiqa",
            Path("/content/verifiqa"),
            Path("/content/drive/MyDrive/verifiqa"),
        ]
    )

    probe = MODEL_SPECS[0].result_path
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            candidates.extend(match.parents[3] for match in base.glob(f"*/{probe}"))

    checked: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(candidate)
        if all(path.is_file() for _, path in model_runs(candidate)):
            return candidate

    listing = "\n".join(f"  - {path}" for path in checked)
    raise FileNotFoundError(
        f"Could not locate all {len(MODEL_SPECS)} XBRLBench result files. "
        f"Set ROOT to the verifiqa repository directory. Checked:\n{listing}"
    )


def load_questions(root: Path) -> dict[str, dict]:
    """Load canonical questions, falling back to text embedded in a run."""
    dataset_path = root / DATASET_RELPATH
    if dataset_path.is_file():
        return {row["financebench_id"]: row for row in read_jsonl(dataset_path)}

    reference_path = model_runs(root)[0][1]
    questions = {
        row["id"]: {
            "financebench_id": row["id"],
            "question": row["question"],
        }
        for row in read_jsonl(reference_path)
        if row.get("id") and row.get("question")
    }
    if len(questions) != N_QUESTIONS:
        raise ValueError(
            f"Expected {N_QUESTIONS} embedded questions in {reference_path}; "
            f"found {len(questions)}."
        )

    warnings.warn(
        f"{dataset_path} was not found; using question text embedded in "
        f"{reference_path}.",
        stacklevel=2,
    )
    return questions


def statement_family(question: str) -> str:
    lowered = question.lower()
    for term, family in FAMILY_TERMS:
        if term in lowered:
            return family
    raise ValueError(f"Unmapped XBRLBench question: {question}")


def parse_numbers(text: object) -> list[float]:
    values: list[float] = []
    for token in _NUMBER_PATTERN.findall(str(text or "")):
        try:
            values.append(abs(float(token.replace(",", ""))))
        except ValueError:
            continue
    return values


def is_correct(claimed_value: object, gold: object) -> bool:
    """Paper convention: magnitude match within max($1m, 1%)."""
    try:
        claimed = abs(float(claimed_value))
    except (TypeError, ValueError):
        return False

    return any(
        abs(claimed - value) <= max(1.0, value * 0.01)
        for value in parse_numbers(gold)
    )


def family_denominators(questions: dict[str, dict]) -> dict[str, int]:
    """Count questions in each statement family."""
    totals = dict.fromkeys(FAMILY_ORDER, 0)
    for row in questions.values():
        totals[statement_family(row["question"])] += 1
    return totals


def tally_run(
    path: Path,
    questions: dict[str, dict],
) -> tuple[dict[str, int], dict[str, int], int, int]:
    """Return caught errors, all errors, false accepts, and unresolved cases."""
    caught = dict.fromkeys(FAMILY_ORDER, 0)
    incorrect = dict.fromkeys(FAMILY_ORDER, 0)
    false_accepts = 0
    unresolved = 0

    rows = read_jsonl(path)
    if len(rows) != N_QUESTIONS:
        raise ValueError(f"{path}: expected {N_QUESTIONS} rows; found {len(rows)}.")

    row_ids = [row.get("id") for row in rows]
    if len(set(row_ids)) != N_QUESTIONS:
        raise ValueError(f"{path}: result IDs are missing or duplicated.")

    unknown_ids = [row_id for row_id in row_ids if row_id not in questions]
    if unknown_ids:
        raise ValueError(f"{path}: {len(unknown_ids)} result IDs are not in the dataset.")

    for row in rows:
        claimed = row.get("claimed_value")
        gold = row.get("gold_answer") or row.get("answer")
        if claimed is None or is_correct(claimed, gold):
            continue

        family = statement_family(questions[row["id"]]["question"])
        incorrect[family] += 1

        status = row.get("status")
        if status == "VIOLATED":
            caught[family] += 1
        elif status == "VERIFIED":
            false_accepts += 1
        else:
            unresolved += 1

    return caught, incorrect, false_accepts, unresolved


# ---------------------------------------------------------------------------
# Figure construction
# ---------------------------------------------------------------------------


def _format_rate(rate: float) -> str:
    """Use one decimal only when rounding to an integer would lose meaning."""
    rounded = round(rate)
    if abs(rate - rounded) < 0.05:
        return f"{rounded:.0f}%"
    return f"{rate:.1f}%" if rate < 10.0 else f"{rounded:.0f}%"


def _add_bar_annotations(
    ax: plt.Axes,
    bars_by_family: dict[str, object],
    rates: pd.DataFrame,
    counts: pd.DataFrame,
) -> None:
    """Place the percentage and raw count together above every bar.

    The full annotation block is anchored to the rectangle's true horizontal
    center and begins a few points above its top edge. Nothing is drawn inside
    the bar, and no leader lines or independently solved coordinates are used.
    """
    for family in FAMILY_ORDER:
        for model, rect in zip(MODEL_ORDER, bars_by_family[family]):
            centre_x = rect.get_x() + rect.get_width() / 2.0
            height = float(rect.get_height())
            rate = float(rates.loc[model, family])
            count = int(counts.loc[model, family])

            ax.annotate(
                f"{_format_rate(rate)}\n({count})",
                xy=(centre_x, height),
                xytext=(0, ANNOTATION_PADDING_PT),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=ANNOTATION_FONTSIZE,
                fontweight="normal",
                linespacing=ANNOTATION_LINESPACING,
                color=INK,
                annotation_clip=False,
                zorder=6,
            )

def make_figure(
    error_counts: pd.DataFrame,
    family_totals: dict[str, int],
    output_stem: Path,
) -> plt.Figure:
    """Create and export the publication figure.

    Parameters
    ----------
    error_counts:
        DataFrame indexed by model with one integer-count column per family.
    family_totals:
        Number of questions in each statement family.
    output_stem:
        Output path without an extension. PDF, SVG, and PNG are written.
    """
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)

    ordered_counts = error_counts.reindex(index=MODEL_ORDER, columns=FAMILY_ORDER)
    if ordered_counts.isna().any().any():
        raise ValueError("error_counts is missing a required model or family.")

    invalid_totals = [family for family in FAMILY_ORDER if family_totals.get(family, 0) <= 0]
    if invalid_totals:
        raise ValueError(f"Missing or invalid family totals: {invalid_totals}")

    rates = pd.DataFrame(
        {
            family: 100.0 * ordered_counts[family] / family_totals[family]
            for family in FAMILY_ORDER
        },
        index=ordered_counts.index,
    )
    with plt.rc_context(
        {
            "font.family": "sans-serif",
            "font.sans-serif": FONT_STACK,
            "font.size": 8.5,
            "axes.linewidth": 0.75,
            "hatch.linewidth": 0.65,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "figure.dpi": 150,
            "savefig.dpi": 600,
        }
    ):
        fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))
        fig.subplots_adjust(left=0.095, right=0.995, bottom=0.215, top=0.785)

        x = np.arange(len(MODEL_ORDER), dtype=float)
        bars_by_family: dict[str, object] = {}

        for (family, fill, edge, hatch, _), offset in zip(
            FAMILY_STYLES,
            BAR_OFFSETS,
        ):
            bars_by_family[family] = ax.bar(
                x + offset,
                rates[family].to_numpy(),
                width=BAR_WIDTH,
                color=fill,
                edgecolor=edge,
                linewidth=0.70,
                hatch=hatch,
                zorder=3,
            )

        ax.set_xlim(-0.50, len(MODEL_ORDER) - 0.50)
        ax.set_ylim(0.0, Y_AXIS_MAX)
        ax.set_yticks(np.arange(0.0, Y_AXIS_MAX + 0.1, Y_TICK_STEP))
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))

        ax.set_ylabel(
            "Wrong Claims Caught by VeriFin (%)",
            fontsize=8.8,
            fontweight="normal",
            color=INK,
            labelpad=6.5,
        )
        ax.set_xticks(x, MODEL_TICK_LABELS)
        ax.tick_params(
            axis="x",
            length=0,
            pad=6.0,
            labelsize=7.9,
            labelcolor=INK,
        )
        ax.tick_params(
            axis="y",
            length=2.8,
            width=0.65,
            pad=3.5,
            labelsize=7.6,
            colors=MUTED,
        )

        ax.set_axisbelow(True)
        ax.grid(axis="y", color=GRID, linewidth=0.55, alpha=1.0)
        ax.grid(axis="x", visible=False)

        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.spines["bottom"].set_color(BASELINE)
        ax.spines["bottom"].set_linewidth(0.75)

        fig.legend(
            handles=[
                Patch(
                    facecolor=fill,
                    edgecolor=edge,
                    linewidth=0.70,
                    hatch=hatch,
                    label=f"{family} (n={family_totals[family]:,})",
                )
                for family, fill, edge, hatch, _ in FAMILY_STYLES
            ],
            loc="upper center",
            bbox_to_anchor=(0.55, 0.985),
            ncol=3,
            frameon=False,
            fontsize=7.9,
            handlelength=1.35,
            handleheight=0.85,
            handletextpad=0.45,
            columnspacing=1.20,
            borderaxespad=0.0,
        )

        _add_bar_annotations(ax, bars_by_family, rates, ordered_counts)

        export_options = {
            "bbox_inches": "tight",
            "pad_inches": 0.025,
            "facecolor": "white",
        }
        fig.savefig(output_stem.with_suffix(".pdf"), **export_options)
        fig.savefig(output_stem.with_suffix(".svg"), **export_options)
        fig.savefig(output_stem.with_suffix(".png"), dpi=600, **export_options)
        return fig


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------


def main(root: Path) -> plt.Figure:
    questions = load_questions(root)
    totals = family_denominators(questions)

    caught_rows: list[dict] = []
    incorrect_rows: list[dict] = []
    false_accepts: dict[str, int] = {}
    unresolved: dict[str, int] = {}

    for model, path in model_runs(root):
        caught, incorrect, fa_count, unresolved_count = tally_run(path, questions)
        caught_rows.append({"Answer model": model, **caught})
        incorrect_rows.append({"Answer model": model, **incorrect})
        false_accepts[model] = fa_count
        unresolved[model] = unresolved_count

    caught_df = pd.DataFrame(caught_rows).set_index("Answer model")
    incorrect_df = pd.DataFrame(incorrect_rows).set_index("Answer model")

    if not caught_df[FAMILY_ORDER].equals(incorrect_df[FAMILY_ORDER]):
        raise AssertionError("At least one incorrect answer was not flagged.")
    if any(false_accepts.values()):
        raise AssertionError(f"False accepts found: {false_accepts}")
    if any(unresolved.values()):
        raise AssertionError(f"Unresolved cases found: {unresolved}")
    if caught_df.index.tolist() != MODEL_ORDER:
        raise AssertionError("Model order differs from MODEL_SPECS.")
    if sum(totals.values()) != N_QUESTIONS:
        raise AssertionError(
            f"Expected {N_QUESTIONS} questions; family totals sum to "
            f"{sum(totals.values())}."
        )

    total_flagged = int(caught_df[FAMILY_ORDER].to_numpy().sum())
    print(f"Total incorrect answers flagged: {total_flagged}")

    figure = make_figure(
        incorrect_df[FAMILY_ORDER],
        totals,
        root / "figures/xbrl_error_profile_by_statement_family",
    )
    plt.show()
    return figure


if __name__ == "__main__":
    ROOT = resolve_root()
    _figure = main(ROOT)

In [ ]:
# Transposed single-column heatmap: models are rows; metrics are columns.
# The script reads only the six result artifacts, so the canonical dataset
# JSONL is not required when this notebook is run in Colab.
import runpy

_heatmap_namespace = runpy.run_path(
    str(ROOT / "scripts" / "plot_six_model_accuracy_heatmap.py")
)
six_model_heatmap_data, _six_model_heatmap_figure = _heatmap_namespace[
    "main"
](ROOT)


![Transposed six-model generated-answer accuracy heatmap](../figures/category_model_performance_transposed.png)

**Paper caption.** Generated-answer accuracy on XBRLBench before verification ($n=600$ per model), disaggregated by requested line item. Models are rows; panels separate income-statement, balance-sheet, and cash-flow metrics, with overall accuracy shown beside each model and metric sample sizes in parentheses. Each cell reports the percentage of questions for which the numerical candidate $\hat y$ satisfies $\bigl|\lvert\hat y\rvert-\lvert y\rvert\bigr|\leq\max(\$1\mathrm{M},0.01\lvert y\rvert)$ for gold answer $y$; missing or nonnumerical candidates count as incorrect. This figure measures answer generation, not \textsc{VeriFin}'s verification accuracy.

## Safety--coverage sensitivity -- numerical tolerance (measured)

This section uses the current Qwen3-30B result files for both datasets. It freezes every candidate answer, grounded fact, formula, and gold-correctness label, multiplies each question's configured numerical tolerance by $\kappa$, and re-solves the stored SMT obligation. The plotted operating point is therefore a genuine verifier sensitivity result rather than a reconstructed confidence proxy.

**Scope check.** Numerical tolerance is the only requested control that is both operational and recoverable from the current traces. Concept matching has a hard-coded cutoff but all logged current-run scores are 100; formula authority is categorical rather than scored; and ambiguity is an exact tie/unique-binding rule with no saved runner-up margin. Adding traces for those three controls from these files would invent thresholds that the evaluated system did not use.

In [ ]:
# Plot the locked tolerance-sweep points without re-solving SMT obligations.
# To perform a new replay intentionally, run scripts/safety_coverage_tolerance.py
# in an environment with the Z3 executable available.
import runpy
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

points_path = ROOT / "figures/safety_coverage_tolerance_sweep_points.csv"
safety_coverage_points = pd.read_csv(points_path)
assert set(safety_coverage_points["Dataset"]) == {"XBRLBench", "FinanceBench"}
assert set(safety_coverage_points["Tolerance multiplier"]) == {
    0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0,
    25.0, 50.0, 100.0, 250.0, 1000.0,
}

_safety_namespace = runpy.run_path(
    str(ROOT / "scripts" / "safety_coverage_tolerance.py")
)
_safety_coverage_figure = _safety_namespace["plot_safety_coverage"](
    safety_coverage_points,
    ROOT / "figures/safety_coverage_tolerance_sweep",
)


**Paper caption.** Numerical-tolerance safety--yield sensitivity on XBRLBench and FinanceBench using fixed Qwen3-30B candidate answers. Each point re-solves the same stored SMT obligations after multiplying the question-specific base tolerance by $\kappa$; evidence, formulas, candidates, and correctness labels remain fixed. Correct-accept yield is $\mathrm{TA}/N$, with all $N=600$ and $N=67$ questions retained, and false-accept risk is $\mathrm{FA}/(\mathrm{TA}+\mathrm{FA})$. Diamonds mark the deployed $\kappa=1$ configuration; dashed whiskers give the one-sided exact 95\% upper bound at the zero-observed-FA deployed point. Zero denotes no observed false accepts, not zero population risk.

In [ ]:
"""Plot six-model XBRLBench answer accuracy before VeriFin is applied.

Compact single-column variant for an ACM two-column paper. Transposed
relative to the original: models are rows (6), metrics are columns (16,
including an "Overall" column) -- this cuts figure height roughly in half
versus putting all 15 metrics on separate rows.

No title and no per-column header labels are drawn in-figure (both were
removed on request) -- column identity (Overall, then GP/OI/OpEx/... in
that order) and the figure title now need to live in the LaTeX caption
instead, e.g.:

  Fig. N. Generated-answer accuracy before VeriFin (%). Columns left to
  right: Overall, then GP = gross profit, OI = operating income,
  OpEx = operating expenses, IBT = income before taxes, NI = net income,
  CA = current assets, NCA = non-current assets, CL = current liabilities,
  NCL = non-current liabilities, TL = total liabilities,
  SE = shareholders' equity, PPE = net PP&E, CFO = cash from operations,
  CFI = cash from investing, CFF = cash from financing. n = 40 per metric
  column, n = 600 for "Overall". Colors above each column group indicate
  statement family (see legend).
"""

from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch, Rectangle


N_QUESTIONS = 600

MODEL_SPECS = (
    (
        "GPT-5.5",
        "GPT-5.5",
        "results/latest/mc_calc_gpt5.5_codex_det_smt_Nrepair_run1/results.jsonl",
    ),
    (
        "Claude Haiku 4.5",
        "Haiku 4.5",
        "results/mc_calc_detsmt_v2_haiku/results.jsonl",
    ),
    (
        "Fin-o1",
        "Fin-o1",
        "results/latest/mc-fino-30b-latest/results.jsonl",
    ),
    (
        "Qwen3-30B",
        "Qwen3 30B",
        "results/latest/mc-qwen-30b-latest/results.jsonl",
    ),
    (
        "Llama-3.1-8B",
        "Llama3.1 8B",
        "results/mc_calc_detsmt_v2_llama3.1-8b/results.jsonl",
    ),
    (
        "Qwen2.5-7B",
        "Qwen2.5 7B",
        "results/mc_calc_detsmt_v2_qwen2.5-7b/results.jsonl",
    ),
)

FAMILY_ORDER = ("Income statement", "Balance sheet", "Cash flow")
FAMILY_COLOR = {
    # Sampled from the VeriFin bar-chart figure's legend so both figures in
    # the paper use the same statement-family colors.
    "Income statement": "#30639C",
    "Balance sheet": "#368579",
    "Cash flow": "#C38A3A",
}

# (search_term, full metric label, family, short column code)
METRIC_SPECS = (
    ("gross profit", "Gross profit", "Income statement", "GP"),
    ("operating income", "Operating income", "Income statement", "OI"),
    ("operating expenses", "Operating expenses", "Income statement", "OpEx"),
    ("income before income taxes", "Income before taxes", "Income statement", "IBT"),
    ("net income", "Net income", "Income statement", "NI"),
    ("total current assets", "Current assets", "Balance sheet", "CA"),
    ("total non-current assets", "Non-current assets", "Balance sheet", "NCA"),
    ("total current liabilities", "Current liabilities", "Balance sheet", "CL"),
    ("total non-current liabilities", "Non-current liabilities", "Balance sheet", "NCL"),
    ("total liabilities", "Total liabilities", "Balance sheet", "TL"),
    ("total shareholders", "Shareholders' equity", "Balance sheet", "SE"),
    ("net property", "Net PP&E", "Balance sheet", "PPE"),
    ("net cash from operating", "Cash from operations", "Cash flow", "CFO"),
    ("net cash from investing", "Cash from investing", "Cash flow", "CFI"),
    ("net cash from financing", "Cash from financing", "Cash flow", "CFF"),
)

METRIC_ORDER = tuple(label for _, label, _, _ in METRIC_SPECS)
METRIC_FAMILY = {label: family for _, label, family, _ in METRIC_SPECS}
METRIC_CODE = {label: code for _, label, _, code in METRIC_SPECS}
MODEL_ORDER = tuple(name for name, _, _ in MODEL_SPECS)
MODEL_ROW_LABELS = tuple(label for _, label, _ in MODEL_SPECS)

NUMBER_PATTERN = re.compile(r"-?\d[\d,]*\.?\d*")

INK = "#20202A"
MUTED = "#666674"
SEPARATOR = "#8A8494"
HEATMAP_CMAP = LinearSegmentedColormap.from_list(
    "candidate_accuracy_purple",
    ("#F7F5FA", "#DDD5E8", "#AA91C3", "#765394", "#4B2D68"),
)

# ACM sigconf single-column width is ~3.33in.
FIG_WIDTH_IN = 3.35
HEATMAP_HEIGHT_IN = 1.93  # just the 6x16 grid + a hair of top margin
LEGEND_HEIGHT_IN = 0.24  # just the 3-item family-color key
FIG_HEIGHT_IN = HEATMAP_HEIGHT_IN + LEGEND_HEIGHT_IN

PNG_DPI = 1200

# The Overall/"All" column shows one decimal place (see _format_pct) and
# needs more horizontal room than the 15 whole-number metric columns to fit
# values like "99.5" without touching the cell edges.
OVERALL_COLUMN_WIDTH_FACTOR = 1.6


def read_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def result_paths(root: Path) -> tuple[tuple[str, str, Path], ...]:
    return tuple(
        (name, tick_label, root / relative_path)
        for name, tick_label, relative_path in MODEL_SPECS
    )


def resolve_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = (
        cwd,
        *cwd.parents,
        cwd / "verifiqa",
        Path("/content/verifiqa"),
        Path("/content/drive/MyDrive/verifiqa"),
    )
    for candidate in candidates:
        if all(path.is_file() for _, _, path in result_paths(candidate)):
            return candidate
    raise FileNotFoundError(
        "Could not locate all six result files. Run from the verifiqa "
        "repository or call main(ROOT) from the notebook."
    )


def metric_for_question(question: str) -> tuple[str, str]:
    lowered = str(question).lower()
    for search_term, metric, family, _ in METRIC_SPECS:
        if search_term in lowered:
            return metric, family
    raise ValueError(f"Unmapped XBRLBench question: {question}")


def parse_numbers(text: object) -> list[float]:
    values: list[float] = []
    for token in NUMBER_PATTERN.findall(str(text or "")):
        try:
            values.append(abs(float(token.replace(",", ""))))
        except ValueError:
            continue
    return values


def is_correct(claimed_value: object, gold_answer: object) -> bool:
    try:
        claimed = abs(float(claimed_value))
    except (TypeError, ValueError):
        return False
    return any(
        abs(claimed - gold) <= max(1.0, 0.01 * gold)
        for gold in parse_numbers(gold_answer)
    )


def load_accuracy_data(root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    runs: dict[str, dict[str, dict]] = {}
    reference_ids: set[str] | None = None
    reference_metric_by_id: dict[str, str] | None = None

    for model, _, path in result_paths(root):
        if not path.is_file():
            raise FileNotFoundError(path)

        rows = read_jsonl(path)
        if len(rows) != N_QUESTIONS:
            raise ValueError(f"{path}: expected {N_QUESTIONS} rows; found {len(rows)}.")

        rows_by_id = {row.get("id"): row for row in rows}
        if None in rows_by_id or len(rows_by_id) != N_QUESTIONS:
            raise ValueError(f"{path}: result IDs are missing or duplicated.")

        ids = set(rows_by_id)
        metric_by_id = {
            row_id: metric_for_question(row.get("question", ""))[0]
            for row_id, row in rows_by_id.items()
        }
        if reference_ids is None:
            reference_ids = ids
            reference_metric_by_id = metric_by_id
        else:
            if ids != reference_ids:
                raise ValueError(f"{path}: IDs do not match the reference run.")
            if metric_by_id != reference_metric_by_id:
                raise ValueError(
                    f"{path}: question-to-metric mapping differs from the reference run."
                )

        missing_gold = [rid for rid, row in rows_by_id.items() if not row.get("gold_answer")]
        if missing_gold:
            raise ValueError(f"{path}: {len(missing_gold)} rows lack gold_answer.")
        runs[model] = rows_by_id

    assert reference_metric_by_id is not None

    denominators = (
        pd.Series(reference_metric_by_id, name="metric")
        .value_counts()
        .reindex(METRIC_ORDER)
        .astype(int)
    )
    if denominators.isna().any() or int(denominators.sum()) != N_QUESTIONS:
        raise AssertionError("Metric denominators do not cover all questions.")

    correct_counts = pd.DataFrame(
        0,
        index=pd.Index(METRIC_ORDER, name="Metric"),
        columns=pd.Index(MODEL_ORDER, name="Answer model"),
        dtype=int,
    )

    for model in MODEL_ORDER:
        for row_id, row in runs[model].items():
            metric = reference_metric_by_id[row_id]
            correct_counts.loc[metric, model] += int(
                is_correct(row.get("claimed_value"), row["gold_answer"])
            )

    accuracy_pct = correct_counts.div(denominators, axis=0) * 100.0

    if not ((correct_counts >= 0).all().all()):
        raise AssertionError("Negative correct count.")
    for metric in METRIC_ORDER:
        if (correct_counts.loc[metric] > denominators.loc[metric]).any():
            raise AssertionError(f"Correct count exceeds n for {metric}.")

    return accuracy_pct, correct_counts, denominators


def _format_pct(value: float) -> str:
    # Rounded to whole numbers -- 15 narrow columns don't have room for a
    # decimal place. Exact values still live in the exported CSV.
    return f"{value:.0f}"


def source_data_frame(
    accuracy_pct: pd.DataFrame,
    correct_counts: pd.DataFrame,
    denominators: pd.Series,
) -> pd.DataFrame:
    records: list[dict] = []
    for model, _, relative_path in MODEL_SPECS:
        records.append(
            {
                "family": "Overall",
                "metric": "Overall",
                "model": model,
                "correct": int(correct_counts[model].sum()),
                "n": N_QUESTIONS,
                "accuracy_pct": 100.0 * float(correct_counts[model].sum()) / N_QUESTIONS,
                "result_path": relative_path,
            }
        )
        for metric in METRIC_ORDER:
            records.append(
                {
                    "family": METRIC_FAMILY[metric],
                    "metric": metric,
                    "model": model,
                    "correct": int(correct_counts.loc[metric, model]),
                    "n": int(denominators.loc[metric]),
                    "accuracy_pct": float(accuracy_pct.loc[metric, model]),
                    "result_path": relative_path,
                }
            )
    return pd.DataFrame.from_records(records)


def _draw_legend(fig: plt.Figure, *, top: float) -> None:
    """Draw the family-color key (Income statement / Balance sheet / Cash
    flow), centered, in a dedicated axes spanning the figure width from
    y=0 up to ``top`` (figure-fraction coordinates). Column headers stay
    as short codes (GP, OI, ...); spell those out in the LaTeX caption if
    needed -- only the three statement-family groupings get an in-figure
    legend."""

    ax = fig.add_axes([0.0, 0.0, 1.0, top])
    ax.axis("off")

    handles = [
        Patch(facecolor=FAMILY_COLOR[family], edgecolor="none")
        for family in FAMILY_ORDER
    ]
    ax.legend(
        handles,
        FAMILY_ORDER,
        loc="center",
        ncol=3,
        frameon=False,
        fontsize=5.1,
        handlelength=1.1,
        handleheight=1.0,
        columnspacing=1.3,
        handletextpad=0.5,
        borderaxespad=0.0,
        labelcolor=INK,
    )


def make_figure(
    accuracy_pct: pd.DataFrame,
    correct_counts: pd.DataFrame,
    denominators: pd.Series,
    output_stem: Path,
) -> plt.Figure:
    """Compact heatmap: models on y (6 rows), Overall + 15 metric codes on x
    (16 columns). Family membership shown as a slim colored bar above the
    column headers instead of row-height-consuming labels."""

    matrix = accuracy_pct.T.reindex(columns=METRIC_ORDER).reindex(index=list(MODEL_ORDER)).to_numpy(
        dtype=float
    )
    n_cols = matrix.shape[1]
    n_rows = matrix.shape[0]

    with plt.rc_context(
        {
            "font.family": "sans-serif",
            "font.sans-serif": [
                "Liberation Sans",
                "Arial",
                "Helvetica Neue",
                "Helvetica",
                "DejaVu Sans",
            ],
            "font.size": 6.0,
            "axes.linewidth": 0.6,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "path",
            "figure.dpi": 150,
        }
    ):
        fig = plt.figure(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN), facecolor="white")

        # No header row / title anymore -- just enough top margin for the
        # family-color bar (which draws slightly above the axes) to avoid
        # being clipped by the figure edge.
        header_in = 0.09
        legend_gap_in = 0.12
        top = 1.0 - header_in / FIG_HEIGHT_IN
        heatmap_bottom = (LEGEND_HEIGHT_IN + legend_gap_in) / FIG_HEIGHT_IN
        ax = fig.add_axes([0.185, heatmap_bottom, 0.995 - 0.185, top - heatmap_bottom])

        norm = Normalize(vmin=0.0, vmax=100.0)
        for row in range(n_rows):
            for column in range(n_cols):
                value = matrix[row, column]
                ax.add_patch(
                    Rectangle(
                        (column - 0.5, row - 0.5),
                        1.0,
                        1.0,
                        facecolor=HEATMAP_CMAP(norm(value)),
                        edgecolor="none",
                    )
                )

        for row in range(n_rows):
            for column in range(n_cols):
                value = matrix[row, column]
                ax.text(
                    column,
                    row,
                    _format_pct(value),
                    ha="center",
                    va="center",
                    color="white" if value >= 66.0 else INK,
                    fontsize=5.1,
                    fontweight="semibold",
                )

        ax.set_xticks([])

        ax.set_yticks(np.arange(n_rows))
        ax.set_yticklabels(MODEL_ROW_LABELS, fontsize=5.8, color=INK)
        ax.tick_params(axis="y", length=0, pad=3.5)

        ax.set_xticks(np.arange(-0.5, n_cols, 1.0), minor=True)
        ax.set_yticks(np.arange(-0.5, n_rows, 1.0), minor=True)
        ax.grid(which="minor", color="white", linewidth=0.9)
        ax.tick_params(which="minor", bottom=False, left=False)

        # Family column boundaries (columns re-index from 0 now that the
        # Overall/All column is gone).
        family_bounds = {
            "Income statement": (-0.5, 4.5),
            "Balance sheet": (4.5, 11.5),
            "Cash flow": (11.5, 14.5),
        }
        for boundary, linewidth in ((4.5, 0.9), (11.5, 0.9)):
            ax.axvline(boundary, color=SEPARATOR, linewidth=linewidth, alpha=0.92, zorder=5)

        # Slim colored family bar sitting just above the column-code header.
        for family, (left, right) in family_bounds.items():
            ax.add_patch(
                Rectangle(
                    (left, -0.5),
                    right - left,
                    -0.22,
                    facecolor=FAMILY_COLOR[family],
                    edgecolor="none",
                    clip_on=False,
                    transform=ax.transData,
                )
            )

        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.add_patch(
            Rectangle(
                (-0.5, -0.5),
                n_cols,
                n_rows,
                fill=False,
                edgecolor="#D0CBD6",
                linewidth=0.6,
                clip_on=False,
            )
        )
        ax.set_xlim(-0.5, n_cols - 0.5)
        ax.set_ylim(n_rows - 0.5, -0.5)

        _draw_legend(fig, top=heatmap_bottom - 0.01)

        output_stem = Path(output_stem)
        output_stem.parent.mkdir(parents=True, exist_ok=True)
        export_options = {"facecolor": "white"}
        fig.savefig(output_stem.with_suffix(".pdf"), **export_options)
        fig.savefig(output_stem.with_suffix(".svg"), **export_options)
        fig.savefig(output_stem.with_suffix(".png"), dpi=PNG_DPI, **export_options)
        return fig


def main(root: Path, *, show: bool = True) -> tuple[pd.DataFrame, plt.Figure]:
    root = Path(root).expanduser().resolve()
    accuracy_pct, correct_counts, denominators = load_accuracy_data(root)
    source_data = source_data_frame(accuracy_pct, correct_counts, denominators)

    output_stem = root / "figures/category_model_performance_1col_compact"
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    source_data.to_csv(
        root / "figures/category_model_performance_1col_compact_data.csv",
        index=False,
        float_format="%.6f",
    )
    figure = make_figure(accuracy_pct, correct_counts, denominators, output_stem)

    overall = (
        source_data[source_data["metric"] == "Overall"]
        .set_index("model")["accuracy_pct"]
        .reindex(MODEL_ORDER)
    )
    print("Overall generated-answer accuracy before VeriFin:")
    for model, value in overall.items():
        print(f"  {model}: {value:.1f}%")
    print(f"Saved {output_stem.with_suffix('.pdf')}")
    print(f"Saved {output_stem.with_suffix('.svg')}")
    print(f"Saved {output_stem.with_suffix('.png')}")
    print(f"Saved {root / 'figures/category_model_performance_1col_compact_data.csv'}")

    if show:
        plt.show()
    return source_data, figure


if __name__ == "__main__":
    main(resolve_root())

In [ ]:
"""Plot six-model XBRLBench answer accuracy before VeriFin is applied.

Compact single-column variant for an ACM two-column paper. Transposed
relative to the original: models are rows (6), metrics are columns (16,
including an "Overall" column) -- this cuts figure height roughly in half
versus putting all 15 metrics on separate rows.

No title and no per-column header labels are drawn in-figure (both were
removed on request) -- column identity (Overall, then GP/OI/OpEx/... in
that order) and the figure title now need to live in the LaTeX caption
instead, e.g.:

  Fig. N. Generated-answer accuracy before VeriFin (%). Columns left to
  right: Overall, then GP = gross profit, OI = operating income,
  OpEx = operating expenses, IBT = income before taxes, NI = net income,
  CA = current assets, NCA = non-current assets, CL = current liabilities,
  NCL = non-current liabilities, TL = total liabilities,
  SE = shareholders' equity, PPE = net PP&E, CFO = cash from operations,
  CFI = cash from investing, CFF = cash from financing. n = 40 per metric
  column, n = 600 for "Overall". Colors above each column group indicate
  statement family (see legend).
"""

from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch, Rectangle


N_QUESTIONS = 600

MODEL_SPECS = (
    (
        "GPT-5.5",
        "GPT-5.5",
        "results/latest/mc_calc_gpt5.5_codex_det_smt_Nrepair_run1/results.jsonl",
    ),
    (
        "Claude Haiku 4.5",
        "Haiku 4.5",
        "results/mc_calc_detsmt_v2_haiku/results.jsonl",
    ),
    (
        "Fin-o1",
        "Fin-o1",
        "results/latest/mc-fino-30b-latest/results.jsonl",
    ),
    (
        "Qwen3-30B",
        "Qwen3 30B",
        "results/latest/mc-qwen-30b-latest/results.jsonl",
    ),
    (
        "Llama-3.1-8B",
        "Llama3.1 8B",
        "results/mc_calc_detsmt_v2_llama3.1-8b/results.jsonl",
    ),
    (
        "Qwen2.5-7B",
        "Qwen2.5 7B",
        "results/mc_calc_detsmt_v2_qwen2.5-7b/results.jsonl",
    ),
)

FAMILY_ORDER = ("Income statement", "Balance sheet", "Cash flow")
FAMILY_COLOR = {
    # Sampled from the VeriFin bar-chart figure's legend so both figures in
    # the paper use the same statement-family colors.
    "Income statement": "#30639C",
    "Balance sheet": "#368579",
    "Cash flow": "#C38A3A",
}

# (search_term, full metric label, family, short column code)
METRIC_SPECS = (
    ("gross profit", "Gross profit", "Income statement", "GP"),
    ("operating income", "Operating income", "Income statement", "OI"),
    ("operating expenses", "Operating expenses", "Income statement", "OpEx"),
    ("income before income taxes", "Income before taxes", "Income statement", "IBT"),
    ("net income", "Net income", "Income statement", "NI"),
    ("total current assets", "Current assets", "Balance sheet", "CA"),
    ("total non-current assets", "Non-current assets", "Balance sheet", "NCA"),
    ("total current liabilities", "Current liabilities", "Balance sheet", "CL"),
    ("total non-current liabilities", "Non-current liabilities", "Balance sheet", "NCL"),
    ("total liabilities", "Total liabilities", "Balance sheet", "TL"),
    ("total shareholders", "Shareholders' equity", "Balance sheet", "SE"),
    ("net property", "Net PP&E", "Balance sheet", "PPE"),
    ("net cash from operating", "Cash from operations", "Cash flow", "CFO"),
    ("net cash from investing", "Cash from investing", "Cash flow", "CFI"),
    ("net cash from financing", "Cash from financing", "Cash flow", "CFF"),
)

METRIC_ORDER = tuple(label for _, label, _, _ in METRIC_SPECS)
METRIC_FAMILY = {label: family for _, label, family, _ in METRIC_SPECS}
METRIC_CODE = {label: code for _, label, _, code in METRIC_SPECS}
MODEL_ORDER = tuple(name for name, _, _ in MODEL_SPECS)
MODEL_ROW_LABELS = tuple(label for _, label, _ in MODEL_SPECS)

NUMBER_PATTERN = re.compile(r"-?\d[\d,]*\.?\d*")

INK = "#20202A"
MUTED = "#666674"
SEPARATOR = "#8A8494"
HEATMAP_CMAP = LinearSegmentedColormap.from_list(
    "candidate_accuracy_purple",
    ("#F7F5FA", "#DDD5E8", "#AA91C3", "#765394", "#4B2D68"),
)

# ACM sigconf single-column width is ~3.33in.
FIG_WIDTH_IN = 3.35
HEATMAP_HEIGHT_IN = 2.75  # 6x16 grid + room for near-vertical per-metric labels
LEGEND_HEIGHT_IN = 0.24  # just the 3-item family-color key
FIG_HEIGHT_IN = HEATMAP_HEIGHT_IN + LEGEND_HEIGHT_IN

PNG_DPI = 1200

# The Overall/"All" column shows one decimal place (see _format_pct) and
# needs more horizontal room than the 15 whole-number metric columns to fit
# values like "99.5" without touching the cell edges.
OVERALL_COLUMN_WIDTH_FACTOR = 1.6


def read_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def result_paths(root: Path) -> tuple[tuple[str, str, Path], ...]:
    return tuple(
        (name, tick_label, root / relative_path)
        for name, tick_label, relative_path in MODEL_SPECS
    )


def resolve_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = (
        cwd,
        *cwd.parents,
        cwd / "verifiqa",
        Path("/content/verifiqa"),
        Path("/content/drive/MyDrive/verifiqa"),
    )
    for candidate in candidates:
        if all(path.is_file() for _, _, path in result_paths(candidate)):
            return candidate
    raise FileNotFoundError(
        "Could not locate all six result files. Run from the verifiqa "
        "repository or call main(ROOT) from the notebook."
    )


def metric_for_question(question: str) -> tuple[str, str]:
    lowered = str(question).lower()
    for search_term, metric, family, _ in METRIC_SPECS:
        if search_term in lowered:
            return metric, family
    raise ValueError(f"Unmapped XBRLBench question: {question}")


def parse_numbers(text: object) -> list[float]:
    values: list[float] = []
    for token in NUMBER_PATTERN.findall(str(text or "")):
        try:
            values.append(abs(float(token.replace(",", ""))))
        except ValueError:
            continue
    return values


def is_correct(claimed_value: object, gold_answer: object) -> bool:
    try:
        claimed = abs(float(claimed_value))
    except (TypeError, ValueError):
        return False
    return any(
        abs(claimed - gold) <= max(1.0, 0.01 * gold)
        for gold in parse_numbers(gold_answer)
    )


def load_accuracy_data(root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    runs: dict[str, dict[str, dict]] = {}
    reference_ids: set[str] | None = None
    reference_metric_by_id: dict[str, str] | None = None

    for model, _, path in result_paths(root):
        if not path.is_file():
            raise FileNotFoundError(path)

        rows = read_jsonl(path)
        if len(rows) != N_QUESTIONS:
            raise ValueError(f"{path}: expected {N_QUESTIONS} rows; found {len(rows)}.")

        rows_by_id = {row.get("id"): row for row in rows}
        if None in rows_by_id or len(rows_by_id) != N_QUESTIONS:
            raise ValueError(f"{path}: result IDs are missing or duplicated.")

        ids = set(rows_by_id)
        metric_by_id = {
            row_id: metric_for_question(row.get("question", ""))[0]
            for row_id, row in rows_by_id.items()
        }
        if reference_ids is None:
            reference_ids = ids
            reference_metric_by_id = metric_by_id
        else:
            if ids != reference_ids:
                raise ValueError(f"{path}: IDs do not match the reference run.")
            if metric_by_id != reference_metric_by_id:
                raise ValueError(
                    f"{path}: question-to-metric mapping differs from the reference run."
                )

        missing_gold = [rid for rid, row in rows_by_id.items() if not row.get("gold_answer")]
        if missing_gold:
            raise ValueError(f"{path}: {len(missing_gold)} rows lack gold_answer.")
        runs[model] = rows_by_id

    assert reference_metric_by_id is not None

    denominators = (
        pd.Series(reference_metric_by_id, name="metric")
        .value_counts()
        .reindex(METRIC_ORDER)
        .astype(int)
    )
    if denominators.isna().any() or int(denominators.sum()) != N_QUESTIONS:
        raise AssertionError("Metric denominators do not cover all questions.")

    correct_counts = pd.DataFrame(
        0,
        index=pd.Index(METRIC_ORDER, name="Metric"),
        columns=pd.Index(MODEL_ORDER, name="Answer model"),
        dtype=int,
    )

    for model in MODEL_ORDER:
        for row_id, row in runs[model].items():
            metric = reference_metric_by_id[row_id]
            correct_counts.loc[metric, model] += int(
                is_correct(row.get("claimed_value"), row["gold_answer"])
            )

    accuracy_pct = correct_counts.div(denominators, axis=0) * 100.0

    if not ((correct_counts >= 0).all().all()):
        raise AssertionError("Negative correct count.")
    for metric in METRIC_ORDER:
        if (correct_counts.loc[metric] > denominators.loc[metric]).any():
            raise AssertionError(f"Correct count exceeds n for {metric}.")

    return accuracy_pct, correct_counts, denominators


def _format_pct(value: float) -> str:
    # Rounded to whole numbers -- 15 narrow columns don't have room for a
    # decimal place. Exact values still live in the exported CSV.
    return f"{value:.0f}"


def source_data_frame(
    accuracy_pct: pd.DataFrame,
    correct_counts: pd.DataFrame,
    denominators: pd.Series,
) -> pd.DataFrame:
    records: list[dict] = []
    for model, _, relative_path in MODEL_SPECS:
        records.append(
            {
                "family": "Overall",
                "metric": "Overall",
                "model": model,
                "correct": int(correct_counts[model].sum()),
                "n": N_QUESTIONS,
                "accuracy_pct": 100.0 * float(correct_counts[model].sum()) / N_QUESTIONS,
                "result_path": relative_path,
            }
        )
        for metric in METRIC_ORDER:
            records.append(
                {
                    "family": METRIC_FAMILY[metric],
                    "metric": metric,
                    "model": model,
                    "correct": int(correct_counts.loc[metric, model]),
                    "n": int(denominators.loc[metric]),
                    "accuracy_pct": float(accuracy_pct.loc[metric, model]),
                    "result_path": relative_path,
                }
            )
    return pd.DataFrame.from_records(records)


def _draw_legend(fig: plt.Figure, *, top: float) -> None:
    """Draw the family-color key (Income statement / Balance sheet / Cash
    flow), centered, in a dedicated axes spanning the figure width from
    y=0 up to ``top`` (figure-fraction coordinates). Column headers stay
    as short codes (GP, OI, ...); spell those out in the LaTeX caption if
    needed -- only the three statement-family groupings get an in-figure
    legend."""

    ax = fig.add_axes([0.0, 0.0, 1.0, top])
    ax.axis("off")

    handles = [
        Patch(facecolor=FAMILY_COLOR[family], edgecolor="none")
        for family in FAMILY_ORDER
    ]
    ax.legend(
        handles,
        FAMILY_ORDER,
        loc="center",
        ncol=3,
        frameon=False,
        fontsize=5.1,
        handlelength=1.1,
        handleheight=1.0,
        columnspacing=1.3,
        handletextpad=0.5,
        borderaxespad=0.0,
        labelcolor=INK,
    )


def make_figure(
    accuracy_pct: pd.DataFrame,
    correct_counts: pd.DataFrame,
    denominators: pd.Series,
    output_stem: Path,
) -> plt.Figure:
    """Compact heatmap: models on y (6 rows), Overall + 15 metric codes on x
    (16 columns). Family membership shown as a slim colored bar above the
    column headers instead of row-height-consuming labels."""

    matrix = accuracy_pct.T.reindex(columns=METRIC_ORDER).reindex(index=list(MODEL_ORDER)).to_numpy(
        dtype=float
    )
    n_cols = matrix.shape[1]
    n_rows = matrix.shape[0]

    with plt.rc_context(
        {
            "font.family": "sans-serif",
            "font.sans-serif": [
                "Liberation Sans",
                "Arial",
                "Helvetica Neue",
                "Helvetica",
                "DejaVu Sans",
            ],
            "font.size": 6.0,
            "axes.linewidth": 0.6,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "path",
            "figure.dpi": 150,
        }
    ):
        fig = plt.figure(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN), facecolor="white")

        # Room at top for the diagonal family-name labels sitting above the
        # color bar (plus a hair of margin so the bar itself isn't clipped).
        header_in = 0.91
        legend_gap_in = 0.12
        top = 1.0 - header_in / FIG_HEIGHT_IN
        heatmap_bottom = (LEGEND_HEIGHT_IN + legend_gap_in) / FIG_HEIGHT_IN
        ax = fig.add_axes([0.185, heatmap_bottom, 0.995 - 0.185, top - heatmap_bottom])

        norm = Normalize(vmin=0.0, vmax=100.0)
        for row in range(n_rows):
            for column in range(n_cols):
                value = matrix[row, column]
                ax.add_patch(
                    Rectangle(
                        (column - 0.5, row - 0.5),
                        1.0,
                        1.0,
                        facecolor=HEATMAP_CMAP(norm(value)),
                        edgecolor="none",
                    )
                )

        for row in range(n_rows):
            for column in range(n_cols):
                value = matrix[row, column]
                ax.text(
                    column,
                    row,
                    _format_pct(value),
                    ha="center",
                    va="center",
                    color="white" if value >= 66.0 else INK,
                    fontsize=5.1,
                    fontweight="semibold",
                )

        ax.set_xticks([])

        ax.set_yticks(np.arange(n_rows))
        ax.set_yticklabels(MODEL_ROW_LABELS, fontsize=5.8, color=INK)
        ax.tick_params(axis="y", length=0, pad=3.5)

        ax.set_xticks(np.arange(-0.5, n_cols, 1.0), minor=True)
        ax.set_yticks(np.arange(-0.5, n_rows, 1.0), minor=True)
        ax.grid(which="minor", color="white", linewidth=0.9)
        ax.tick_params(which="minor", bottom=False, left=False)

        # Family column boundaries (columns re-index from 0 now that the
        # Overall/All column is gone).
        family_bounds = {
            "Income statement": (-0.5, 4.5),
            "Balance sheet": (4.5, 11.5),
            "Cash flow": (11.5, 14.5),
        }
        for boundary, linewidth in ((4.5, 0.9), (11.5, 0.9)):
            ax.axvline(boundary, color=SEPARATOR, linewidth=linewidth, alpha=0.92, zorder=5)

        # Slim colored family bar sitting just above the grid.
        for family, (left, right) in family_bounds.items():
            ax.add_patch(
                Rectangle(
                    (left, -0.5),
                    right - left,
                    -0.22,
                    facecolor=FAMILY_COLOR[family],
                    edgecolor="none",
                    clip_on=False,
                    transform=ax.transData,
                )
            )

        # Each individual metric name, written diagonally above its own
        # column, colored by which statement family it belongs to.
        for column, metric in enumerate(METRIC_ORDER):
            ax.text(
                column - 0.5 + 0.08,
                -0.78,
                metric,
                ha="left",
                va="bottom",
                rotation=83,
                rotation_mode="anchor",
                fontsize=4.4,
                fontweight="semibold",
                color=FAMILY_COLOR[METRIC_FAMILY[metric]],
                clip_on=False,
            )

        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.add_patch(
            Rectangle(
                (-0.5, -0.5),
                n_cols,
                n_rows,
                fill=False,
                edgecolor="#D0CBD6",
                linewidth=0.6,
                clip_on=False,
            )
        )
        ax.set_xlim(-0.5, n_cols - 0.5)
        ax.set_ylim(n_rows - 0.5, -0.5)

        _draw_legend(fig, top=heatmap_bottom - 0.01)

        output_stem = Path(output_stem)
        output_stem.parent.mkdir(parents=True, exist_ok=True)
        export_options = {"facecolor": "white"}
        fig.savefig(output_stem.with_suffix(".pdf"), **export_options)
        fig.savefig(output_stem.with_suffix(".svg"), **export_options)
        fig.savefig(output_stem.with_suffix(".png"), dpi=PNG_DPI, **export_options)
        return fig


def main(root: Path, *, show: bool = True) -> tuple[pd.DataFrame, plt.Figure]:
    root = Path(root).expanduser().resolve()
    accuracy_pct, correct_counts, denominators = load_accuracy_data(root)
    source_data = source_data_frame(accuracy_pct, correct_counts, denominators)

    output_stem = root / "figures/category_model_performance_1col_compact"
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    source_data.to_csv(
        root / "figures/category_model_performance_1col_compact_data.csv",
        index=False,
        float_format="%.6f",
    )
    figure = make_figure(accuracy_pct, correct_counts, denominators, output_stem)

    overall = (
        source_data[source_data["metric"] == "Overall"]
        .set_index("model")["accuracy_pct"]
        .reindex(MODEL_ORDER)
    )
    print("Overall generated-answer accuracy before VeriFin:")
    for model, value in overall.items():
        print(f"  {model}: {value:.1f}%")
    print(f"Saved {output_stem.with_suffix('.pdf')}")
    print(f"Saved {output_stem.with_suffix('.svg')}")
    print(f"Saved {output_stem.with_suffix('.png')}")
    print(f"Saved {root / 'figures/category_model_performance_1col_compact_data.csv'}")

    if show:
        plt.show()
    return source_data, figure


if __name__ == "__main__":
    main(resolve_root())